In [1]:
# Import Path để tạo đường dẫn đến các file dữ liệu
from pathlib import Path
import pandas as pd
import numpy as np

# Import công cụ dùng để tạo chuỗi các bước tiền xử lý
from sklearn.pipeline import Pipeline

# Import công cụ dùng để điền các giá trị bị thiếu
from sklearn.impute import SimpleImputer

# Import công cụ dùng để chuyển dữ liệu phân loại thành các cột 0 và 1
from sklearn.preprocessing import OneHotEncoder

# Import công cụ dùng để áp dụng cách xử lý khác nhau cho từng nhóm cột
from sklearn.compose import ColumnTransformer

In [2]:
# Tạo đường dẫn đến dataset gốc
original_data_path = Path(
    "../data/dataset_anonymization.csv"
)

# Tạo đường dẫn đến dataset ứng viên đã đạt k=5 và l=2
anonymized_data_path = Path(
    "../data/processed_da_xu_li/anonymized_candidate_k5_l2.csv"
)


# Đọc dataset gốc vào RAM
data_original = pd.read_csv(
    original_data_path,
    encoding="utf-8-sig"
)

# Đọc dataset đã ẩn danh vào RAM
data_anonymized_ml = pd.read_csv(
    anonymized_data_path,
    encoding="utf-8-sig"
)


In [3]:
# Hiển thị kích thước của hai dataset để kiểm tra đã đọc đúng chưa
print(
    "Kích thước dữ liệu gốc:",
    data_original.shape
)

print(
    "Kích thước dữ liệu ẩn danh:",
    data_anonymized_ml.shape
)

Kích thước dữ liệu gốc: (32561, 15)
Kích thước dữ liệu ẩn danh: (30604, 8)


In [4]:
# Tạo đường dẫn đến file chứa chỉ số các bản ghi được giữ lại sau ẩn danh
retained_indices_path = Path(
    "../data/processed_da_xu_li/retained_row_indices_k5_l2.csv"
)


# Đọc danh sách chỉ số của 30.604 bản ghi còn lại
retained_indices_df = pd.read_csv(
    retained_indices_path,
    encoding="utf-8-sig"
)


# Lấy riêng cột chỉ số và chuyển về kiểu số nguyên
retained_indices = (
    retained_indices_df["original_row_index"]
    .astype(int)
)


# Lấy đúng các bản ghi tương ứng từ dataset gốc
# Như vậy dữ liệu gốc và dữ liệu ẩn danh sẽ đại diện cho cùng một tập bản ghi
data_original_matched = (
    data_original
    .loc[retained_indices]
    .copy()
)


# Đặt lại chỉ số từ 0 để sau này dễ chia train/test cùng vị trí
data_original_matched = (
    data_original_matched
    .reset_index(drop=True)
)

data_anonymized_ml = (
    data_anonymized_ml
    .reset_index(drop=True)
)


# Kiểm tra kích thước của hai dataset sau khi ghép đúng bản ghi
print(
    "Dữ liệu gốc đã khớp:",
    data_original_matched.shape
)

print(
    "Dữ liệu ẩn danh:",
    data_anonymized_ml.shape
)

Dữ liệu gốc đã khớp: (30604, 15)
Dữ liệu ẩn danh: (30604, 8)


In [5]:
# Chuẩn hóa nhẹ cột thu_nhap của dữ liệu gốc
# strip() loại bỏ khoảng trắng thừa nếu có
target_original_check = (
    data_original_matched["thu_nhap"]
    .astype(str)
    .str.strip()
)


# Chuẩn hóa nhẹ cột thu_nhap của dữ liệu ẩn danh
target_anonymized_check = (
    data_anonymized_ml["thu_nhap"]
    .astype(str)
    .str.strip()
)


# So sánh thu_nhap của từng dòng giữa hai dataset
target_match_mask = (
    target_original_check == target_anonymized_check
)


# Đếm số dòng có nhãn thu_nhap khớp nhau
so_dong_khop = int(
    target_match_mask.sum()
)


# Đếm số dòng có nhãn thu_nhap không khớp nhau
so_dong_khong_khop = int(
    (~target_match_mask).sum()
)


# Hiển thị kết quả kiểm tra
print(
    "Số dòng có thu_nhap khớp:",
    so_dong_khop
)

print(
    "Số dòng có thu_nhap không khớp:",
    so_dong_khong_khop
)

print(
    "Tỷ lệ khớp:",
    round(target_match_mask.mean() * 100, 2),
    "%"
)

Số dòng có thu_nhap khớp: 30604
Số dòng có thu_nhap không khớp: 0
Tỷ lệ khớp: 100.0 %


In [6]:
# Chọn 7 thuộc tính gốc tương ứng với 7 QI đã dùng trong dữ liệu ẩn danh
original_feature_columns = [
    "tuoi",
    "loai_cong_viec",
    "so_nam_hoc",
    "tinh_trang_hon_nhan",
    "nghe_nghiep",
    "gio_lam_moi_tuan",
    "quoc_gia"
]


# Chọn 7 thuộc tính đã được generalization trong dữ liệu ẩn danh
anonymized_feature_columns = [
    "tuoi_nhom_3",
    "loai_cong_viec_nhom",
    "hoc_van_nhom_3",
    "hon_nhan_nhom_2",
    "nghe_nghiep_nhom_4",
    "gio_lam_nhom_3",
    "quoc_gia_nhom_2"
]


# Tạo tập đặc trưng X của dữ liệu gốc
X_original = (
    data_original_matched[
        original_feature_columns
    ]
    .copy()
)


# Tạo tập đặc trưng X của dữ liệu ẩn danh
X_anonymized = (
    data_anonymized_ml[
        anonymized_feature_columns
    ]
    .copy()
)


# Lấy cột thu_nhap làm nhãn cần dự đoán
# Hai dataset đã được kiểm tra khớp 100% nên chỉ cần lấy một lần
y = (
    data_original_matched["thu_nhap"]
    .astype(str)
    .str.strip()
)


# Kiểm tra kích thước của hai tập đặc trưng và nhãn
print("X_original:", X_original.shape)
print("X_anonymized:", X_anonymized.shape)
print("y:", y.shape)

X_original: (30604, 7)
X_anonymized: (30604, 7)
y: (30604,)


Chia chung train/validation/test

In [7]:
# Import hàm dùng để chia dữ liệu thành các tập train, validation và test
from sklearn.model_selection import train_test_split

In [8]:
# Lấy danh sách vị trí của toàn bộ 30.604 bản ghi
# Ta chia chỉ số thay vì chia trực tiếp X để cả dữ liệu gốc và ẩn danh dùng cùng một cách chia
all_indices = X_original.index


# Chia lần thứ nhất:
# 60% bản ghi dùng để train
# 40% còn lại tạm thời dành cho validation và test
train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.40,
    random_state=42,
    stratify=y
)


# Chia 40% còn lại thành hai phần bằng nhau
# Kết quả cuối cùng là 20% validation và 20% test
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=y.loc[temp_indices]
)


# Hiển thị số lượng bản ghi của từng tập
print(
    "Số bản ghi Train:",
    len(train_indices)
)

print(
    "Số bản ghi Validation:",
    len(val_indices)
)

print(
    "Số bản ghi Test:",
    len(test_indices)
)

print(
    "Tổng:",
    len(train_indices)
    + len(val_indices)
    + len(test_indices)
)

Số bản ghi Train: 18362
Số bản ghi Validation: 6121
Số bản ghi Test: 6121
Tổng: 30604


In [9]:
# Tạo tập Train của dữ liệu gốc
# Chỉ lấy những dòng thuộc train_indices
X_original_train = (
    X_original
    .loc[train_indices]
    .copy()
)


# Tạo tập Validation của dữ liệu gốc
X_original_val = (
    X_original
    .loc[val_indices]
    .copy()
)


# Tạo tập Test của dữ liệu gốc
X_original_test = (
    X_original
    .loc[test_indices]
    .copy()
)


# Tạo tập Train của dữ liệu ẩn danh
# Sử dụng cùng train_indices với dữ liệu gốc để đảm bảo so sánh công bằng
X_anonymized_train = (
    X_anonymized
    .loc[train_indices]
    .copy()
)


# Tạo tập Validation của dữ liệu ẩn danh
X_anonymized_val = (
    X_anonymized
    .loc[val_indices]
    .copy()
)


# Tạo tập Test của dữ liệu ẩn danh
X_anonymized_test = (
    X_anonymized
    .loc[test_indices]
    .copy()
)


# Tạo nhãn thu_nhap cho tập Train
y_train = (
    y
    .loc[train_indices]
    .copy()
)


# Tạo nhãn thu_nhap cho tập Validation
y_val = (
    y
    .loc[val_indices]
    .copy()
)


# Tạo nhãn thu_nhap cho tập Test
y_test = (
    y
    .loc[test_indices]
    .copy()
)


# Kiểm tra kích thước các tập Train của dữ liệu gốc và dữ liệu ẩn danh
print("X_original_train:", X_original_train.shape)
print("X_anonymized_train:", X_anonymized_train.shape)
print("y_train:", y_train.shape)

# Kiểm tra kích thước Validation
print("\nX_original_val:", X_original_val.shape)
print("X_anonymized_val:", X_anonymized_val.shape)
print("y_val:", y_val.shape)

# Kiểm tra kích thước Test
print("\nX_original_test:", X_original_test.shape)
print("X_anonymized_test:", X_anonymized_test.shape)
print("y_test:", y_test.shape)

X_original_train: (18362, 7)
X_anonymized_train: (18362, 7)
y_train: (18362,)

X_original_val: (6121, 7)
X_anonymized_val: (6121, 7)
y_val: (6121,)

X_original_test: (6121, 7)
X_anonymized_test: (6121, 7)
y_test: (6121,)


In [10]:
# Tạo bảng để kiểm tra dữ liệu thiếu trong tập Train của dữ liệu gốc
missing_original_train = pd.DataFrame({
    # Đếm số giá trị NaN thực sự trong từng cột
    "so_nan": X_original_train.isna().sum(),

    # Đếm số giá trị được ghi bằng dấu ? trong từng cột
    "so_dau_hoi": (X_original_train == "?").sum()
})


# Tính tổng số giá trị thiếu của từng cột
missing_original_train["tong_thieu"] = (
    missing_original_train["so_nan"]
    + missing_original_train["so_dau_hoi"]
)


# Chỉ hiển thị những cột thực sự có dữ liệu thiếu
missing_original_train = (
    missing_original_train[
        missing_original_train["tong_thieu"] > 0
    ]
)


# Hiển thị kết quả kiểm tra dữ liệu thiếu trong tập Train
missing_original_train

,so_nan,so_dau_hoi,tong_thieu
loai_cong_viec,0,733,733
nghe_nghiep,0,733,733
quoc_gia,0,257,257


Kiểm tra dấu ? và N xuất hiện trong dữ liệu


In [11]:
# Tạo bảng để kiểm tra dữ liệu thiếu trong tập Train của dữ liệu gốc
missing_original_train = pd.DataFrame({
    # Đếm số giá trị NaN thực sự trong từng cột
    "so_nan": X_original_train.isna().sum(),

    # Đếm số giá trị được ghi bằng dấu ? trong từng cột
    "so_dau_hoi": (X_original_train == "?").sum()
})


# Tính tổng số giá trị thiếu của từng cột
missing_original_train["tong_thieu"] = (
    missing_original_train["so_nan"]
    + missing_original_train["so_dau_hoi"]
)


# Chỉ hiển thị những cột thực sự có dữ liệu thiếu
missing_original_train = (
    missing_original_train[
        missing_original_train["tong_thieu"] > 0
    ]
)


# Hiển thị kết quả kiểm tra dữ liệu thiếu trong tập Train
missing_original_train

,so_nan,so_dau_hoi,tong_thieu
loai_cong_viec,0,733,733
nghe_nghiep,0,733,733
quoc_gia,0,257,257


In [12]:
# Thay dấu ? bằng pd.NA trong tập Train của dữ liệu gốc
# pd.NA là cách pandas biểu diễn giá trị bị thiếu
X_original_train = (
    X_original_train
    .replace("?", pd.NA)
)


# Thay dấu ? bằng pd.NA trong tập Validation
# Bước này không học thông tin từ Validation nên không gây data leakage
X_original_val = (
    X_original_val
    .replace("?", pd.NA)
)


# Thay dấu ? bằng pd.NA trong tập Test
# Chỉ chuẩn hóa ký hiệu dữ liệu thiếu, chưa thực hiện điền giá trị
X_original_test = (
    X_original_test
    .replace("?", pd.NA)
)


# Kiểm tra lại số lượng giá trị thiếu trong tập Train
# Sau bước này các giá trị ? phải được tính thành NaN
print(
    X_original_train
    .isna()
    .sum()
)

tuoi                     0
loai_cong_viec         733
so_nam_hoc               0
tinh_trang_hon_nhan      0
nghe_nghiep            733
gio_lam_moi_tuan         0
quoc_gia               257
dtype: int64


Trong tập Train dữ  liệu hiên có

loai_cong_viec    733 giá trị thiếu

nghe_nghiep       733 giá trị thiếu

quoc_gia          257 giá trị thiếu

Cột số

tuoi, so_nam_hoc, gio_lam_moi_tuan

        ↓
Điền missing bằng median

        ↓
StandardScaler

Cột phân loại

loai_cong_viec, tinh_trang_hon_nhan, nghe_nghiep, quoc_gia

        ↓
Điền missing bằng giá trị xuất hiện nhiều nhất

        ↓
One-Hot Encoding

In [13]:
# Import công cụ điền dữ liệu thiếu cho cột số
from sklearn.impute import SimpleImputer

# Import công cụ chuẩn hóa dữ liệu số
from sklearn.preprocessing import StandardScaler

# Import Pipeline để ghép nhiều bước tiền xử lý lại với nhau
from sklearn.pipeline import Pipeline


# Tạo lại pipeline xử lý các cột số của dữ liệu gốc
original_numeric_pipeline = Pipeline(
    steps=[
        # Nếu có dữ liệu số bị thiếu thì điền bằng giá trị trung vị
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        # Chuẩn hóa các cột số trước khi đưa vào mô hình
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Kiểm tra biến đã được tạo thành công
print(original_numeric_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [14]:
# Khai báo lại 3 cột số của dữ liệu gốc
original_numeric_columns = [
    "tuoi",
    "so_nam_hoc",
    "gio_lam_moi_tuan"
]


# Khai báo lại 4 cột phân loại của dữ liệu gốc
original_categorical_columns = [
    "loai_cong_viec",
    "tinh_trang_hon_nhan",
    "nghe_nghiep",
    "quoc_gia"
]


# Khai báo lại 7 cột phân loại của dữ liệu ẩn danh
anonymized_categorical_columns = [
    "tuoi_nhom_3",
    "loai_cong_viec_nhom",
    "hoc_van_nhom_3",
    "hon_nhan_nhom_2",
    "nghe_nghiep_nhom_4",
    "gio_lam_nhom_3",
    "quoc_gia_nhom_2"
]


# Kiểm tra các danh sách cột đã được tạo lại
print("Cột số dữ liệu gốc:", original_numeric_columns)
print("Cột phân loại dữ liệu gốc:", original_categorical_columns)
print("Cột phân loại dữ liệu ẩn danh:", anonymized_categorical_columns)

Cột số dữ liệu gốc: ['tuoi', 'so_nam_hoc', 'gio_lam_moi_tuan']
Cột phân loại dữ liệu gốc: ['loai_cong_viec', 'tinh_trang_hon_nhan', 'nghe_nghiep', 'quoc_gia']
Cột phân loại dữ liệu ẩn danh: ['tuoi_nhom_3', 'loai_cong_viec_nhom', 'hoc_van_nhom_3', 'hon_nhan_nhom_2', 'nghe_nghiep_nhom_4', 'gio_lam_nhom_3', 'quoc_gia_nhom_2']


In [15]:
# Chuyển pd.NA thành np.nan trong tập Train
# np.nan tương thích ổn định hơn với SimpleImputer
X_original_train = (
    X_original_train
    .replace({pd.NA: np.nan})
)


# Chuyển pd.NA thành np.nan trong tập Validation
X_original_val = (
    X_original_val
    .replace({pd.NA: np.nan})
)


# Chuyển pd.NA thành np.nan trong tập Test
X_original_test = (
    X_original_test
    .replace({pd.NA: np.nan})
)


# Tạo lại pipeline cho các cột phân loại của dữ liệu gốc
original_categorical_pipeline = Pipeline(
    steps=[
        # Điền dữ liệu thiếu bằng giá trị xuất hiện nhiều nhất trong tập Train
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        # Chuyển dữ liệu chữ thành các cột số 0 và 1
        # Nếu Validation/Test có giá trị mới thì bỏ qua để không gây lỗi
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# Ghép lại phần xử lý cột số và cột phân loại
original_preprocessor = ColumnTransformer(
    transformers=[
        # Xử lý 3 cột số
        (
            "numeric",
            original_numeric_pipeline,
            original_numeric_columns
        ),

        # Xử lý 4 cột phân loại
        (
            "categorical",
            original_categorical_pipeline,
            original_categorical_columns
        )
    ]
)


# Dùng print để kiểm tra pipeline đã được tạo thành công
print(original_preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tuoi', 'so_nam_hoc', 'gio_lam_moi_tuan']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['loai_cong_viec', 'tinh_trang_hon_nhan',
                                  'nghe_nghiep', 'quoc_gia'])])


In [16]:
# Tạo pipeline xử lý 7 cột phân loại của dữ liệu ẩn danh
anonymized_categorical_pipeline = Pipeline(
    steps=[
        # Nếu có dữ liệu thiếu thì điền bằng giá trị xuất hiện nhiều nhất trong tập Train
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        # Chuyển các nhóm dạng chữ thành các cột số 0 và 1
        # handle_unknown="ignore" giúp Validation/Test không bị lỗi nếu có nhóm khác
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# Tạo bộ tiền xử lý hoàn chỉnh cho dữ liệu ẩn danh
# Cả 7 cột đều được xử lý như dữ liệu phân loại
anonymized_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            anonymized_categorical_pipeline,
            anonymized_categorical_columns
        )
    ]
)


# In cấu trúc pipeline để kiểm tra đã tạo thành công
print(anonymized_preprocessor)

ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['tuoi_nhom_3', 'loai_cong_viec_nhom',
                                  'hoc_van_nhom_3', 'hon_nhan_nhom_2',
                                  'nghe_nghiep_nhom_4', 'gio_lam_nhom_3',
                                  'quoc_gia_nhom_2'])])


In [17]:
#Sự dụng dữ liệu gốc
# Fit bộ tiền xử lý trên tập Train của dữ liệu gốc
# fit_transform() vừa học cách xử lý từ Train, vừa biến đổi Train
X_original_train_processed = (
    original_preprocessor
    .fit_transform(X_original_train)
)


# Chỉ áp dụng quy tắc đã học từ Train cho tập Validation
# Không dùng fit_transform để tránh data leakage
X_original_val_processed = (
    original_preprocessor
    .transform(X_original_val)
)


# Chỉ áp dụng quy tắc đã học từ Train cho tập Test
# Test tuyệt đối không được dùng để học cách tiền xử lý
X_original_test_processed = (
    original_preprocessor
    .transform(X_original_test)
)


# Fit bộ tiền xử lý riêng trên tập Train của dữ liệu ẩn danh
X_anonymized_train_processed = (
    anonymized_preprocessor
    .fit_transform(X_anonymized_train)
)


# Áp dụng quy tắc đã học từ Train cho Validation của dữ liệu ẩn danh
X_anonymized_val_processed = (
    anonymized_preprocessor
    .transform(X_anonymized_val)
)


# Áp dụng quy tắc đã học từ Train cho Test của dữ liệu ẩn danh
X_anonymized_test_processed = (
    anonymized_preprocessor
    .transform(X_anonymized_test)
)


# Kiểm tra kích thước dữ liệu gốc sau tiền xử lý
print("DỮ LIỆU GỐC SAU TIỀN XỬ LÝ:")
print("Train:", X_original_train_processed.shape)
print("Validation:", X_original_val_processed.shape)
print("Test:", X_original_test_processed.shape)


# Kiểm tra kích thước dữ liệu ẩn danh sau tiền xử lý
print("\nDỮ LIỆU ẨN DANH SAU TIỀN XỬ LÝ:")
print("Train:", X_anonymized_train_processed.shape)
print("Validation:", X_anonymized_val_processed.shape)
print("Test:", X_anonymized_test_processed.shape)

DỮ LIỆU GỐC SAU TIỀN XỬ LÝ:
Train: (18362, 71)
Validation: (6121, 71)
Test: (6121, 71)

DỮ LIỆU ẨN DANH SAU TIỀN XỬ LÝ:
Train: (18362, 21)
Validation: (6121, 21)
Test: (6121, 21)


In [18]:
# Import mô hình Logistic Regression
# Đây sẽ là mô hình baseline đầu tiên để đánh giá khả năng dự đoán thu_nhap
from sklearn.linear_model import LogisticRegression

In [19]:
# Import các chỉ số dùng để đánh giá mô hình
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# Tạo mô hình Logistic Regression cho dữ liệu gốc
# max_iter=1000 giúp mô hình có đủ số vòng lặp để hội tụ
original_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Huấn luyện mô hình chỉ bằng tập Train
original_logistic_model.fit(
    X_original_train_processed,
    y_train
)


# Dự đoán thu_nhap trên tập Validation
y_val_pred_original = original_logistic_model.predict(
    X_original_val_processed
)


# Tính Accuracy trên tập Validation
accuracy_original = accuracy_score(
    y_val,
    y_val_pred_original
)


# Tính Precision cho nhóm thu_nhap >50K
precision_original = precision_score(
    y_val,
    y_val_pred_original,
    pos_label=">50K"
)


# Tính Recall cho nhóm thu_nhap >50K
recall_original = recall_score(
    y_val,
    y_val_pred_original,
    pos_label=">50K"
)


# Tính F1-score cho nhóm thu_nhap >50K
f1_original = f1_score(
    y_val,
    y_val_pred_original,
    pos_label=">50K"
)


# Hiển thị kết quả của mô hình trên dữ liệu gốc
print("LOGISTIC REGRESSION - DỮ LIỆU GỐC")
print("Accuracy:", round(accuracy_original, 4))
print("Precision:", round(precision_original, 4))
print("Recall:", round(recall_original, 4))
print("F1-score:", round(f1_original, 4))

LOGISTIC REGRESSION - DỮ LIỆU GỐC
Accuracy: 0.8223
Precision: 0.6858
Recall: 0.5435
F1-score: 0.6064


Accuracy = 82.23%: tổng thể mô hình dự đoán đúng khoảng 82/100 trường hợp.

Precision = 68.58%: trong những người mô hình dự đoán là >50K, khoảng 68.58% thực sự >50K.

Recall = 54.35%: trong toàn bộ người thực sự >50K, mô hình tìm đúng khoảng 54.35%.

F1 = 60.64%: chỉ số cân bằng giữa Precision và Recall.

In [20]:
#Sự dụng dữ liệu đã ẩn danh
# Tạo một mô hình Logistic Regression mới cho dữ liệu ẩn danh
# Dùng cùng tham số với mô hình dữ liệu gốc để so sánh công bằng
anonymized_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Huấn luyện mô hình bằng tập Train của dữ liệu ẩn danh
anonymized_logistic_model.fit(
    X_anonymized_train_processed,
    y_train
)


# Dự đoán thu_nhap trên đúng tập Validation đã dùng cho dữ liệu gốc
y_val_pred_anonymized = anonymized_logistic_model.predict(
    X_anonymized_val_processed
)


# Tính Accuracy của mô hình trên dữ liệu ẩn danh
accuracy_anonymized = accuracy_score(
    y_val,
    y_val_pred_anonymized
)


# Tính Precision cho nhóm thu_nhap >50K
precision_anonymized = precision_score(
    y_val,
    y_val_pred_anonymized,
    pos_label=">50K"
)


# Tính Recall cho nhóm thu_nhap >50K
recall_anonymized = recall_score(
    y_val,
    y_val_pred_anonymized,
    pos_label=">50K"
)


# Tính F1-score cho nhóm thu_nhap >50K
f1_anonymized = f1_score(
    y_val,
    y_val_pred_anonymized,
    pos_label=">50K"
)


# Hiển thị kết quả của Logistic Regression trên dữ liệu ẩn danh
print("LOGISTIC REGRESSION - DỮ LIỆU ẨN DANH")
print("Accuracy:", round(accuracy_anonymized, 4))
print("Precision:", round(precision_anonymized, 4))
print("Recall:", round(recall_anonymized, 4))
print("F1-score:", round(f1_anonymized, 4))

LOGISTIC REGRESSION - DỮ LIỆU ẨN DANH
Accuracy: 0.8149
Precision: 0.6815
Recall: 0.4981
F1-score: 0.5755


In [21]:
# Tạo bảng so sánh kết quả Logistic Regression
# giữa dữ liệu gốc và dữ liệu đã ẩn danh
logistic_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],

    # Các chỉ số của mô hình huấn luyện bằng dữ liệu gốc
    "Original": [
        accuracy_original,
        precision_original,
        recall_original,
        f1_original
    ],

    # Các chỉ số của mô hình huấn luyện bằng dữ liệu ẩn danh
    "Anonymized": [
        accuracy_anonymized,
        precision_anonymized,
        recall_anonymized,
        f1_anonymized
    ]
})


# Tính mức thay đổi sau khi ẩn danh
# Giá trị âm nghĩa là hiệu năng bị giảm
logistic_comparison["Difference"] = (
    logistic_comparison["Anonymized"]
    - logistic_comparison["Original"]
)


# Chuyển các giá trị sang phần trăm để bảng dễ đọc
logistic_comparison[
    ["Original", "Anonymized", "Difference"]
] = (
    logistic_comparison[
        ["Original", "Anonymized", "Difference"]
    ] * 100
).round(2)


# Hiển thị bảng so sánh
logistic_comparison

,Metric,Original,Anonymized,Difference
0,Accuracy,82.23,81.49,-0.74
1,Precision,68.58,68.15,-0.43
2,Recall,54.35,49.81,-4.54
3,F1-score,60.64,57.55,-3.09


In [22]:
# Import Balanced Accuracy và ROC-AUC để đánh giá mô hình đầy đủ hơn
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score
)


# Tính Balanced Accuracy của Logistic Regression trên dữ liệu gốc
balanced_accuracy_original = balanced_accuracy_score(
    y_val,
    y_val_pred_original
)


# Tính Balanced Accuracy của Logistic Regression trên dữ liệu ẩn danh
balanced_accuracy_anonymized = balanced_accuracy_score(
    y_val,
    y_val_pred_anonymized
)


# Tìm vị trí của lớp >50K trong mô hình dữ liệu gốc
# Làm như vậy để lấy đúng xác suất dự đoán cho lớp >50K
positive_class_index_original = list(
    original_logistic_model.classes_
).index(">50K")


# Lấy xác suất dự đoán lớp >50K trên Validation của dữ liệu gốc
y_val_prob_original = original_logistic_model.predict_proba(
    X_original_val_processed
)[:, positive_class_index_original]


# Tìm vị trí của lớp >50K trong mô hình dữ liệu ẩn danh
positive_class_index_anonymized = list(
    anonymized_logistic_model.classes_
).index(">50K")


# Lấy xác suất dự đoán lớp >50K trên Validation của dữ liệu ẩn danh
y_val_prob_anonymized = anonymized_logistic_model.predict_proba(
    X_anonymized_val_processed
)[:, positive_class_index_anonymized]


# Tính ROC-AUC của mô hình trên dữ liệu gốc
roc_auc_original = roc_auc_score(
    (y_val == ">50K").astype(int),
    y_val_prob_original
)


# Tính ROC-AUC của mô hình trên dữ liệu ẩn danh
roc_auc_anonymized = roc_auc_score(
    (y_val == ">50K").astype(int),
    y_val_prob_anonymized
)


# Hiển thị kết quả để so sánh
print("LOGISTIC REGRESSION - BỔ SUNG CHỈ SỐ")

print(
    "Balanced Accuracy - Original:",
    round(balanced_accuracy_original, 4)
)

print(
    "Balanced Accuracy - Anonymized:",
    round(balanced_accuracy_anonymized, 4)
)

print(
    "ROC-AUC - Original:",
    round(roc_auc_original, 4)
)

print(
    "ROC-AUC - Anonymized:",
    round(roc_auc_anonymized, 4)
)

LOGISTIC REGRESSION - BỔ SUNG CHỈ SỐ
Balanced Accuracy - Original: 0.7298
Balanced Accuracy - Anonymized: 0.7098
ROC-AUC - Original: 0.8753
ROC-AUC - Anonymized: 0.8611


In [23]:
# Tạo lại bảng so sánh Logistic Regression với đầy đủ 6 chỉ số đánh giá
logistic_comparison_full = pd.DataFrame({
    # Tên các chỉ số đánh giá mô hình
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced Accuracy",
        "ROC-AUC"
    ],

    # Kết quả của mô hình sử dụng dữ liệu gốc
    "Original": [
        accuracy_original,
        precision_original,
        recall_original,
        f1_original,
        balanced_accuracy_original,
        roc_auc_original
    ],

    # Kết quả của mô hình sử dụng dữ liệu đã ẩn danh
    "Anonymized": [
        accuracy_anonymized,
        precision_anonymized,
        recall_anonymized,
        f1_anonymized,
        balanced_accuracy_anonymized,
        roc_auc_anonymized
    ]
})


# Tính mức thay đổi hiệu năng sau khi ẩn danh
# Giá trị âm nghĩa là chỉ số của dữ liệu ẩn danh thấp hơn dữ liệu gốc
logistic_comparison_full["Difference"] = (
    logistic_comparison_full["Anonymized"]
    - logistic_comparison_full["Original"]
)


# Chuyển toàn bộ kết quả sang đơn vị phần trăm để dễ đọc
logistic_comparison_full[
    ["Original", "Anonymized", "Difference"]
] = (
    logistic_comparison_full[
        ["Original", "Anonymized", "Difference"]
    ] * 100
).round(2)


# Hiển thị bảng kết quả Logistic Regression đầy đủ
logistic_comparison_full

,Metric,Original,Anonymized,Difference
0,Accuracy,82.23,81.49,-0.74
1,Precision,68.58,68.15,-0.43
2,Recall,54.35,49.81,-4.54
3,F1-score,60.64,57.55,-3.09
4,Balanced Accuracy,72.98,70.98,-2.00
5,ROC-AUC,87.53,86.11,-1.42


Random Forest

Random Forest là một thuật toán Machine Learning dùng nhiều cây quyết định cùng lúc để đưa ra dự đoá

In [24]:
#Train Ramdom Forest trên dữu lệu gốc
# Import mô hình Random Forest
# Đây là mô hình thứ hai dùng để kiểm tra ảnh hưởng của ẩn danh trên một thuật toán khác
from sklearn.ensemble import RandomForestClassifier


# Tạo mô hình Random Forest cho dữ liệu gốc
# n_estimators=200 nghĩa là mô hình sử dụng 200 cây quyết định
# random_state=42 giúp kết quả có thể tái lập khi chạy lại
# n_jobs=-1 cho phép sử dụng nhiều lõi CPU để train nhanh hơn
original_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


# Huấn luyện Random Forest bằng tập Train của dữ liệu gốc
original_rf_model.fit(
    X_original_train_processed,
    y_train
)


# Dự đoán thu_nhap trên tập Validation
y_val_pred_rf_original = original_rf_model.predict(
    X_original_val_processed
)


# Tính Accuracy trên tập Validation
accuracy_rf_original = accuracy_score(
    y_val,
    y_val_pred_rf_original
)


# Tính Precision cho lớp thu_nhap >50K
precision_rf_original = precision_score(
    y_val,
    y_val_pred_rf_original,
    pos_label=">50K"
)


# Tính Recall cho lớp thu_nhap >50K
recall_rf_original = recall_score(
    y_val,
    y_val_pred_rf_original,
    pos_label=">50K"
)


# Tính F1-score cho lớp thu_nhap >50K
f1_rf_original = f1_score(
    y_val,
    y_val_pred_rf_original,
    pos_label=">50K"
)


# Tính Balanced Accuracy để đánh giá cân bằng giữa hai lớp thu nhập
balanced_accuracy_rf_original = balanced_accuracy_score(
    y_val,
    y_val_pred_rf_original
)


# Tìm vị trí của lớp >50K trong danh sách lớp của mô hình
positive_class_index_rf_original = list(
    original_rf_model.classes_
).index(">50K")


# Lấy xác suất dự đoán thuộc lớp >50K
y_val_prob_rf_original = original_rf_model.predict_proba(
    X_original_val_processed
)[:, positive_class_index_rf_original]


# Tính ROC-AUC từ xác suất dự đoán của lớp >50K
roc_auc_rf_original = roc_auc_score(
    (y_val == ">50K").astype(int),
    y_val_prob_rf_original
)


# Hiển thị kết quả Random Forest trên dữ liệu gốc
print("RANDOM FOREST - DỮ LIỆU GỐC")
print("Accuracy:", round(accuracy_rf_original, 4))
print("Precision:", round(precision_rf_original, 4))
print("Recall:", round(recall_rf_original, 4))
print("F1-score:", round(f1_rf_original, 4))
print(
    "Balanced Accuracy:",
    round(balanced_accuracy_rf_original, 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_rf_original, 4)
)

RANDOM FOREST - DỮ LIỆU GỐC
Accuracy: 0.8074
Precision: 0.6324
Recall: 0.5623
F1-score: 0.5953
Balanced Accuracy: 0.7261
ROC-AUC: 0.8534


So với Logistic Regression trên dữ liệu gốc, Random Forest hiện có Accuracy thấp hơn một chút nhưng Recall cao hơn. Chưa cần kết luận model nào tốt hơn, vì mục tiêu hiện tại là xem cùng một model bị ảnh hưởng thế nào sau ẩn danh.

In [25]:
#Traim Ramdom Forest trên dữu liệu ẩn danh
# Tạo mô hình Random Forest mới cho dữ liệu ẩn danh
# Dùng đúng các tham số giống mô hình dữ liệu gốc để so sánh công bằng
anonymized_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


# Huấn luyện Random Forest bằng tập Train của dữ liệu ẩn danh
anonymized_rf_model.fit(
    X_anonymized_train_processed,
    y_train
)


# Dự đoán thu_nhap trên đúng tập Validation
y_val_pred_rf_anonymized = anonymized_rf_model.predict(
    X_anonymized_val_processed
)


# Tính Accuracy của Random Forest trên dữ liệu ẩn danh
accuracy_rf_anonymized = accuracy_score(
    y_val,
    y_val_pred_rf_anonymized
)


# Tính Precision cho lớp thu_nhap >50K
precision_rf_anonymized = precision_score(
    y_val,
    y_val_pred_rf_anonymized,
    pos_label=">50K"
)


# Tính Recall cho lớp thu_nhap >50K
recall_rf_anonymized = recall_score(
    y_val,
    y_val_pred_rf_anonymized,
    pos_label=">50K"
)


# Tính F1-score cho lớp thu_nhap >50K
f1_rf_anonymized = f1_score(
    y_val,
    y_val_pred_rf_anonymized,
    pos_label=">50K"
)


# Tính Balanced Accuracy để đánh giá cân bằng giữa hai lớp
balanced_accuracy_rf_anonymized = balanced_accuracy_score(
    y_val,
    y_val_pred_rf_anonymized
)


# Tìm vị trí của lớp >50K trong danh sách lớp của mô hình
positive_class_index_rf_anonymized = list(
    anonymized_rf_model.classes_
).index(">50K")


# Lấy xác suất dự đoán thuộc lớp >50K
y_val_prob_rf_anonymized = anonymized_rf_model.predict_proba(
    X_anonymized_val_processed
)[:, positive_class_index_rf_anonymized]


# Tính ROC-AUC của Random Forest trên dữ liệu ẩn danh
roc_auc_rf_anonymized = roc_auc_score(
    (y_val == ">50K").astype(int),
    y_val_prob_rf_anonymized
)


# Hiển thị toàn bộ kết quả Random Forest trên dữ liệu ẩn danh
print("RANDOM FOREST - DỮ LIỆU ẨN DANH")
print("Accuracy:", round(accuracy_rf_anonymized, 4))
print("Precision:", round(precision_rf_anonymized, 4))
print("Recall:", round(recall_rf_anonymized, 4))
print("F1-score:", round(f1_rf_anonymized, 4))
print(
    "Balanced Accuracy:",
    round(balanced_accuracy_rf_anonymized, 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_rf_anonymized, 4)
)


RANDOM FOREST - DỮ LIỆU ẨN DANH
Accuracy: 0.8111
Precision: 0.6771
Recall: 0.4786
F1-score: 0.5608
Balanced Accuracy: 0.7009
ROC-AUC: 0.8533


In [26]:
# Tạo bảng so sánh Random Forest giữa dữ liệu gốc và dữ liệu ẩn danh
rf_comparison = pd.DataFrame({
    # Tên các chỉ số đánh giá
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced Accuracy",
        "ROC-AUC"
    ],

    # Kết quả Random Forest trên dữ liệu gốc
    "Original": [
        accuracy_rf_original,
        precision_rf_original,
        recall_rf_original,
        f1_rf_original,
        balanced_accuracy_rf_original,
        roc_auc_rf_original
    ],

    # Kết quả Random Forest trên dữ liệu ẩn danh
    "Anonymized": [
        accuracy_rf_anonymized,
        precision_rf_anonymized,
        recall_rf_anonymized,
        f1_rf_anonymized,
        balanced_accuracy_rf_anonymized,
        roc_auc_rf_anonymized
    ]
})


# Tính mức thay đổi sau khi ẩn danh
# Giá trị dương nghĩa là chỉ số tăng, giá trị âm nghĩa là chỉ số giảm
rf_comparison["Difference"] = (
    rf_comparison["Anonymized"]
    - rf_comparison["Original"]
)


# Chuyển kết quả sang phần trăm để dễ đọc
rf_comparison[
    ["Original", "Anonymized", "Difference"]
] = (
    rf_comparison[
        ["Original", "Anonymized", "Difference"]
    ] * 100
).round(2)


# Hiển thị bảng so sánh Random Forest
rf_comparison

,Metric,Original,Anonymized,Difference
0,Accuracy,80.74,81.11,0.38
1,Precision,63.24,67.71,4.47
2,Recall,56.23,47.86,-8.37
3,F1-score,59.53,56.08,-3.45
4,Balanced Accuracy,72.61,70.09,-2.52
5,ROC-AUC,85.34,85.33,-0.01


In [27]:
#Gom cả 2 bảng vào để so sánh
# Tạo bản sao bảng kết quả Logistic Regression
# Thêm tên mô hình để sau này biết mỗi dòng thuộc mô hình nào
logistic_summary = logistic_comparison_full.copy()
logistic_summary["Model"] = "Logistic Regression"


# Tạo bản sao bảng kết quả Random Forest
# Thêm tên mô hình vào từng dòng kết quả
rf_summary = rf_comparison.copy()
rf_summary["Model"] = "Random Forest"


# Ghép kết quả của hai mô hình vào cùng một bảng
ml_utility_comparison = pd.concat(
    [
        logistic_summary,
        rf_summary
    ],
    ignore_index=True
)


# Sắp xếp lại thứ tự cột để tên mô hình nằm ở đầu bảng
ml_utility_comparison = ml_utility_comparison[
    [
        "Model",
        "Metric",
        "Original",
        "Anonymized",
        "Difference"
    ]
]


# Hiển thị bảng tổng hợp Utility trên tập Validation
ml_utility_comparison

,Model,Metric,Original,Anonymized,Difference
0,Logistic Regression,Accuracy,82.23,81.49,-0.74
1,Logistic Regression,Precision,68.58,68.15,-0.43
2,Logistic Regression,Recall,54.35,49.81,-4.54
3,Logistic Regression,F1-score,60.64,57.55,-3.09
4,Logistic Regression,Balanced Accuracy,72.98,70.98,-2.00
5,Logistic Regression,ROC-AUC,87.53,86.11,-1.42
6,Random Forest,Accuracy,80.74,81.11,0.38
7,Random Forest,Precision,63.24,67.71,4.47
8,Random Forest,Recall,56.23,47.86,-8.37
9,Random Forest,F1-score,59.53,56.08,-3.45


“Sau khi ẩn danh, Accuracy của Logistic Regression chỉ giảm 0.74%, còn Random Forest tăng nhẹ 0.38%. ROC-AUC cũng gần như được giữ nguyên, đặc biệt Random Forest chỉ giảm 0.01%. Tuy nhiên Recall giảm rõ, cho thấy quá trình ẩn danh làm mất một phần thông tin cần thiết để nhận diện nhóm thu nhập trên 50K. Nhìn chung dữ liệu sau ẩn danh vẫn giữ được utility tương đối tốt nhưng có sự đánh đổi với khả năng phát hiện lớp >50K.”

In [28]:
# Tạo bảng chỉ lấy kết quả của dữ liệu ẩn danh
# để so sánh trực tiếp hai mô hình trên tập Validation
anonymized_model_comparison = (
    ml_utility_comparison[
        [
            "Model",
            "Metric",
            "Anonymized"
        ]
    ]
    .copy()
)


# Chuyển bảng từ dạng nhiều dòng sang dạng:
# mỗi dòng là một mô hình, mỗi cột là một chỉ số
anonymized_model_comparison = (
    anonymized_model_comparison
    .pivot(
        index="Model",
        columns="Metric",
        values="Anonymized"
    )
    .reset_index()
)


# Sắp xếp các cột theo thứ tự dễ đọc
anonymized_model_comparison = anonymized_model_comparison[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced Accuracy",
        "ROC-AUC"
    ]
]


# Hiển thị bảng so sánh hai mô hình trên dữ liệu ẩn danh
anonymized_model_comparison

Metric,Model,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC
0,Logistic Regression,81.49,68.15,49.81,57.55,70.98,86.11
1,Random Forest,81.11,67.71,47.86,56.08,70.09,85.33


Cho hệ thống tự chọn mô hình tốt nhất

In [29]:
# Sắp xếp các mô hình theo F1-score từ cao xuống thấp
# Nếu F1-score bằng nhau thì xét tiếp Balanced Accuracy và ROC-AUC
model_ranking = (
    anonymized_model_comparison
    .sort_values(
        by=[
            "F1-score",
            "Balanced Accuracy",
            "ROC-AUC"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


# Lấy mô hình đứng đầu bảng xếp hạng
best_ml_model = model_ranking.iloc[0]


# Lưu tên mô hình tốt nhất để sử dụng ở các bước sau
selected_model_name = best_ml_model["Model"]


# Hiển thị mô hình được hệ thống lựa chọn
print(
    "Mô hình tốt nhất trên Validation:",
    selected_model_name
)


# Hiển thị bảng xếp hạng đầy đủ
model_ranking

Mô hình tốt nhất trên Validation: Logistic Regression


Metric,Model,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC
0,Logistic Regression,81.49,68.15,49.81,57.55,70.98,86.11
1,Random Forest,81.11,67.71,47.86,56.08,70.09,85.33


Gộp Train + Validation trước khi đánh giá Test

In [30]:
# Ghép tập Train và Validation của dữ liệu gốc
# Sau khi đã chọn mô hình, ta có thể dùng cả hai tập này để huấn luyện mô hình cuối
X_original_trainval = pd.concat(
    [
        X_original_train,
        X_original_val
    ],
    axis=0
)


# Ghép tập Train và Validation của dữ liệu ẩn danh
# Hai phía vẫn sử dụng đúng cùng các bản ghi
X_anonymized_trainval = pd.concat(
    [
        X_anonymized_train,
        X_anonymized_val
    ],
    axis=0
)


# Ghép nhãn của Train và Validation
y_trainval = pd.concat(
    [
        y_train,
        y_val
    ],
    axis=0
)


# Kiểm tra kích thước tập Train + Validation
print(
    "X_original_trainval:",
    X_original_trainval.shape
)

print(
    "X_anonymized_trainval:",
    X_anonymized_trainval.shape
)

print(
    "y_trainval:",
    y_trainval.shape
)


# Kiểm tra lại tập Test vẫn được giữ nguyên và chưa dùng để huấn luyện
print(
    "\nSố bản ghi Test:",
    len(y_test)
)

X_original_trainval: (24483, 7)
X_anonymized_trainval: (24483, 7)
y_trainval: (24483,)

Số bản ghi Test: 6121


Tiền xử lý lại 80% Train + Validation để huấn luyện mô hình cuối, còn Test chỉ được transform để dùng chấm kết quả cuối cùng

In [31]:
# Import clone để tạo bản sao sạch của các bộ tiền xử lý
# Như vậy không làm thay đổi các preprocessor đã dùng ở giai đoạn Validation trước đó
from sklearn.base import clone


# Tạo bản sao mới của bộ tiền xử lý dữ liệu gốc
final_original_preprocessor = clone(
    original_preprocessor
)


# Tạo bản sao mới của bộ tiền xử lý dữ liệu ẩn danh
final_anonymized_preprocessor = clone(
    anonymized_preprocessor
)


# Cho bộ tiền xử lý dữ liệu gốc học lại trên Train + Validation
# Sau đó biến đổi toàn bộ 24.483 bản ghi Train + Validation
X_original_trainval_processed = (
    final_original_preprocessor
    .fit_transform(X_original_trainval)
)


# Chỉ áp dụng quy tắc đã học từ Train + Validation cho Test
# Không fit trên Test để tránh data leakage
X_original_test_final_processed = (
    final_original_preprocessor
    .transform(X_original_test)
)


# Cho bộ tiền xử lý dữ liệu ẩn danh học lại trên Train + Validation
X_anonymized_trainval_processed = (
    final_anonymized_preprocessor
    .fit_transform(X_anonymized_trainval)
)


# Chỉ transform tập Test của dữ liệu ẩn danh
X_anonymized_test_final_processed = (
    final_anonymized_preprocessor
    .transform(X_anonymized_test)
)


# Kiểm tra kích thước dữ liệu gốc sau tiền xử lý cuối
print("DỮ LIỆU GỐC:")
print(
    "Train + Validation:",
    X_original_trainval_processed.shape
)


print(
    "Test:",
    X_original_test_final_processed.shape
)


# Kiểm tra kích thước dữ liệu ẩn danh sau tiền xử lý cuối
print("\nDỮ LIỆU ẨN DANH:")
print(
    "Train + Validation:",
    X_anonymized_trainval_processed.shape
)
print(
    "Test:",
    X_anonymized_test_final_processed.shape
)

DỮ LIỆU GỐC:
Train + Validation: (24483, 71)
Test: (6121, 71)

DỮ LIỆU ẨN DANH:
Train + Validation: (24483, 21)
Test: (6121, 21)


đánh giá Logistic Regression cuối trên Test của dữ liệu gốc
Train 60%

→ mô hình học

Validation 20%

→ dùng để chọn mô hình tốt hơn

Sau khi chọn Logistic Regression

→ gộp Train + Validation = 80%

→ train lại mô hình cuối cùng

Test 20%

→ đem mô hình cuối cùng đi chấm

In [32]:
# Tạo mô hình Logistic Regression cuối cùng cho dữ liệu gốc
# Dùng cùng cấu hình đã được chọn ở giai đoạn Validation
final_original_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Huấn luyện mô hình bằng toàn bộ tập Train + Validation
# Tổng cộng 24.483 bản ghi được dùng để học
final_original_logistic_model.fit(
    X_original_trainval_processed,
    y_trainval
)


# Dự đoán thu_nhap trên tập Test
# Tập Test chưa được dùng để huấn luyện mô hình
y_test_pred_original = final_original_logistic_model.predict(
    X_original_test_final_processed
)


# Tính Accuracy trên tập Test
test_accuracy_original = accuracy_score(
    y_test,
    y_test_pred_original
)


# Tính Precision cho lớp thu_nhap >50K
test_precision_original = precision_score(
    y_test,
    y_test_pred_original,
    pos_label=">50K"
)


# Tính Recall cho lớp thu_nhap >50K
test_recall_original = recall_score(
    y_test,
    y_test_pred_original,
    pos_label=">50K"
)


# Tính F1-score cho lớp thu_nhap >50K
test_f1_original = f1_score(
    y_test,
    y_test_pred_original,
    pos_label=">50K"
)


# Tính Balanced Accuracy để đánh giá cân bằng giữa hai lớp
test_balanced_accuracy_original = balanced_accuracy_score(
    y_test,
    y_test_pred_original
)


# Tìm vị trí của lớp >50K trong danh sách lớp của mô hình
positive_class_index_final_original = list(
    final_original_logistic_model.classes_
).index(">50K")


# Lấy xác suất dự đoán thuộc lớp >50K trên tập Test
y_test_prob_original = final_original_logistic_model.predict_proba(
    X_original_test_final_processed
)[:, positive_class_index_final_original]


# Tính ROC-AUC trên tập Test
test_roc_auc_original = roc_auc_score(
    (y_test == ">50K").astype(int),
    y_test_prob_original
)


# Hiển thị kết quả cuối cùng trên Test của dữ liệu gốc
print("LOGISTIC REGRESSION - TEST - DỮ LIỆU GỐC")
print("Accuracy:", round(test_accuracy_original, 4))
print("Precision:", round(test_precision_original, 4))
print("Recall:", round(test_recall_original, 4))
print("F1-score:", round(test_f1_original, 4))
print(
    "Balanced Accuracy:",
    round(test_balanced_accuracy_original, 4)
)
print(
    "ROC-AUC:",
    round(test_roc_auc_original, 4)
)

LOGISTIC REGRESSION - TEST - DỮ LIỆU GỐC
Accuracy: 0.8216
Precision: 0.6866
Recall: 0.537
F1-score: 0.6026
Balanced Accuracy: 0.7272
ROC-AUC: 0.8769


đánh giá Logistic Regression cuối trên Test của dữ liệu ẩn danh


In [33]:
# Tạo mô hình Logistic Regression cuối cùng cho dữ liệu ẩn danh
# Dùng cùng cấu hình với mô hình dữ liệu gốc để so sánh công bằng
final_anonymized_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Huấn luyện mô hình bằng toàn bộ Train + Validation của dữ liệu ẩn danh
# Tổng cộng 24.483 bản ghi được dùng để học
final_anonymized_logistic_model.fit(
    X_anonymized_trainval_processed,
    y_trainval
)


# Dự đoán thu_nhap trên đúng tập Test đã dùng cho dữ liệu gốc
y_test_pred_anonymized = final_anonymized_logistic_model.predict(
    X_anonymized_test_final_processed
)


# Tính Accuracy trên tập Test
test_accuracy_anonymized = accuracy_score(
    y_test,
    y_test_pred_anonymized
)


# Tính Precision cho lớp thu_nhap >50K
test_precision_anonymized = precision_score(
    y_test,
    y_test_pred_anonymized,
    pos_label=">50K"
)


# Tính Recall cho lớp thu_nhap >50K
test_recall_anonymized = recall_score(
    y_test,
    y_test_pred_anonymized,
    pos_label=">50K"
)


# Tính F1-score cho lớp thu_nhap >50K
test_f1_anonymized = f1_score(
    y_test,
    y_test_pred_anonymized,
    pos_label=">50K"
)


# Tính Balanced Accuracy trên tập Test
test_balanced_accuracy_anonymized = balanced_accuracy_score(
    y_test,
    y_test_pred_anonymized
)


# Tìm vị trí của lớp >50K trong danh sách lớp của mô hình
positive_class_index_final_anonymized = list(
    final_anonymized_logistic_model.classes_
).index(">50K")


# Lấy xác suất dự đoán thuộc lớp >50K trên tập Test
y_test_prob_anonymized = final_anonymized_logistic_model.predict_proba(
    X_anonymized_test_final_processed
)[:, positive_class_index_final_anonymized]


# Tính ROC-AUC trên tập Test
test_roc_auc_anonymized = roc_auc_score(
    (y_test == ">50K").astype(int),
    y_test_prob_anonymized
)


# Hiển thị kết quả cuối cùng trên Test của dữ liệu ẩn danh
print("LOGISTIC REGRESSION - TEST - DỮ LIỆU ẨN DANH")
print("Accuracy:", round(test_accuracy_anonymized, 4))
print("Precision:", round(test_precision_anonymized, 4))
print("Recall:", round(test_recall_anonymized, 4))
print("F1-score:", round(test_f1_anonymized, 4))
print(
    "Balanced Accuracy:",
    round(test_balanced_accuracy_anonymized, 4)
)
print(
    "ROC-AUC:",
    round(test_roc_auc_anonymized, 4)
)

LOGISTIC REGRESSION - TEST - DỮ LIỆU ẨN DANH
Accuracy: 0.8131
Precision: 0.6786
Recall: 0.4903
F1-score: 0.5693
Balanced Accuracy: 0.706
ROC-AUC: 0.8592


Tạo bản kết quả test chính thức

In [34]:
# Tạo bảng so sánh kết quả cuối cùng trên tập Test
# giữa dữ liệu gốc và dữ liệu đã ẩn danh
test_logistic_comparison = pd.DataFrame({
    # Tên các chỉ số đánh giá mô hình
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced Accuracy",
        "ROC-AUC"
    ],

    # Kết quả cuối trên Test của dữ liệu gốc
    "Original": [
        test_accuracy_original,
        test_precision_original,
        test_recall_original,
        test_f1_original,
        test_balanced_accuracy_original,
        test_roc_auc_original
    ],

    # Kết quả cuối trên Test của dữ liệu ẩn danh
    "Anonymized": [
        test_accuracy_anonymized,
        test_precision_anonymized,
        test_recall_anonymized,
        test_f1_anonymized,
        test_balanced_accuracy_anonymized,
        test_roc_auc_anonymized
    ]
})


# Tính mức thay đổi sau khi ẩn danh
# Giá trị âm nghĩa là hiệu năng bị giảm
test_logistic_comparison["Difference"] = (
    test_logistic_comparison["Anonymized"]
    - test_logistic_comparison["Original"]
)


# Chuyển các kết quả sang đơn vị phần trăm để dễ đọc
test_logistic_comparison[
    ["Original", "Anonymized", "Difference"]
] = (
    test_logistic_comparison[
        ["Original", "Anonymized", "Difference"]
    ] * 100
).round(2)


# Hiển thị bảng kết quả Test cuối cùng
test_logistic_comparison

,Metric,Original,Anonymized,Difference
0,Accuracy,82.16,81.31,-0.85
1,Precision,68.66,67.86,-0.79
2,Recall,53.70,49.03,-4.67
3,F1-score,60.26,56.93,-3.33
4,Balanced Accuracy,72.72,70.60,-2.12
5,ROC-AUC,87.69,85.92,-1.77


In [35]:
# In toàn bộ bảng kết quả Test dưới dạng văn bản
# Cách này giúp xem kết quả khi PyCharm không hiển thị DataFrame
print(
    test_logistic_comparison.to_string(
        index=False
    )
)

           Metric  Original  Anonymized  Difference
         Accuracy     82.16       81.31       -0.85
        Precision     68.66       67.86       -0.79
           Recall     53.70       49.03       -4.67
         F1-score     60.26       56.93       -3.33
Balanced Accuracy     72.72       70.60       -2.12
          ROC-AUC     87.69       85.92       -1.77


In [36]:
# Tạo bản sao bảng kết quả Test để tính mức Utility còn giữ lại
test_utility_retention = (
    test_logistic_comparison
    .copy()
)


# Tính tỷ lệ hiệu năng của dữ liệu ẩn danh so với dữ liệu gốc
# Ví dụ 99% nghĩa là sau ẩn danh vẫn giữ được 99% hiệu năng ban đầu
test_utility_retention["Utility_Retention_%"] = (
    test_utility_retention["Anonymized"]
    / test_utility_retention["Original"]
    * 100
).round(2)


# Chỉ giữ các cột cần thiết để bảng dễ đọc
test_utility_retention = test_utility_retention[
    [
        "Metric",
        "Original",
        "Anonymized",
        "Difference",
        "Utility_Retention_%"
    ]
]


# Hiển thị bảng mức Utility còn giữ lại sau ẩn danh
print(
    test_utility_retention.to_string(
        index=False
    )
)

           Metric  Original  Anonymized  Difference  Utility_Retention_%
         Accuracy     82.16       81.31       -0.85                98.97
        Precision     68.66       67.86       -0.79                98.83
           Recall     53.70       49.03       -4.67                91.30
         F1-score     60.26       56.93       -3.33                94.47
Balanced Accuracy     72.72       70.60       -2.12                97.08
          ROC-AUC     87.69       85.92       -1.77                97.98


In [37]:
# Tạo bản sao bảng kết quả Test để tính mức Utility còn giữ lại
test_utility_retention = (
    test_logistic_comparison
    .copy()
)


# Tính tỷ lệ hiệu năng của dữ liệu ẩn danh so với dữ liệu gốc
# Ví dụ 99% nghĩa là sau ẩn danh vẫn giữ được 99% hiệu năng ban đầu
test_utility_retention["Utility_Retention_%"] = (
    test_utility_retention["Anonymized"]
    / test_utility_retention["Original"]
    * 100
).round(2)


# Chỉ giữ các cột cần thiết để bảng dễ đọc
test_utility_retention = test_utility_retention[
    [
        "Metric",
        "Original",
        "Anonymized",
        "Difference",
        "Utility_Retention_%"
    ]
]


# Hiển thị bảng mức Utility còn giữ lại sau ẩn danh
print(
    test_utility_retention.to_string(
        index=False
    )
)

           Metric  Original  Anonymized  Difference  Utility_Retention_%
         Accuracy     82.16       81.31       -0.85                98.97
        Precision     68.66       67.86       -0.79                98.83
           Recall     53.70       49.03       -4.67                91.30
         F1-score     60.26       56.93       -3.33                94.47
Balanced Accuracy     72.72       70.60       -2.12                97.08
          ROC-AUC     87.69       85.92       -1.77                97.98


In [38]:
# Tạo bảng tổng hợp kết quả Privacy và ML Utility
# của strategy ẩn danh hiện tại
current_strategy_summary = pd.DataFrame({
    # Tên strategy đang được đánh giá
    "Strategy": [
        "Country2_Hours3_Age3_Edu3_k5_l2"
    ],

    # Kết quả Privacy đã kiểm tra trong notebook 01
    "k_min": [
        5
    ],

    # Giá trị l-diversity nhỏ nhất đã đạt
    "l_min": [
        2
    ],

    # Tổng tỷ lệ bản ghi bị suppression
    "Suppression_%": [
        6.01
    ],

    # Kết quả Utility cuối cùng trên tập Test
    "Accuracy_Original_%": [
        test_accuracy_original * 100
    ],

    "Accuracy_Anonymized_%": [
        test_accuracy_anonymized * 100
    ],

    # Tỷ lệ Accuracy còn giữ lại sau ẩn danh
    "Accuracy_Retention_%": [
        test_accuracy_anonymized
        / test_accuracy_original
        * 100
    ],

    # Tỷ lệ F1-score còn giữ lại sau ẩn danh
    "F1_Retention_%": [
        test_f1_anonymized
        / test_f1_original
        * 100
    ],

    # Tỷ lệ Balanced Accuracy còn giữ lại
    "Balanced_Accuracy_Retention_%": [
        test_balanced_accuracy_anonymized
        / test_balanced_accuracy_original
        * 100
    ],

    # Tỷ lệ ROC-AUC còn giữ lại
    "ROC_AUC_Retention_%": [
        test_roc_auc_anonymized
        / test_roc_auc_original
        * 100
    ]
})


# Làm tròn các cột số để bảng dễ đọc
current_strategy_summary = (
    current_strategy_summary
    .round(2)
)


# Hiển thị bảng tổng hợp của strategy hiện tại
print(
    current_strategy_summary.to_string(
        index=False
    )
)

                       Strategy  k_min  l_min  Suppression_%  Accuracy_Original_%  Accuracy_Anonymized_%  Accuracy_Retention_%  F1_Retention_%  Balanced_Accuracy_Retention_%  ROC_AUC_Retention_%
Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01                82.16                  81.31                 98.97           94.47                          97.09                97.98


In [39]:
# Lấy tỷ lệ Utility còn giữ lại của Accuracy
accuracy_retention = (
    test_accuracy_anonymized
    / test_accuracy_original
    * 100
)


# Lấy tỷ lệ Utility còn giữ lại của F1-score
f1_retention = (
    test_f1_anonymized
    / test_f1_original
    * 100
)


# Lấy tỷ lệ Utility còn giữ lại của Balanced Accuracy
balanced_accuracy_retention = (
    test_balanced_accuracy_anonymized
    / test_balanced_accuracy_original
    * 100
)


# Lấy tỷ lệ Utility còn giữ lại của ROC-AUC
roc_auc_retention = (
    test_roc_auc_anonymized
    / test_roc_auc_original
    * 100
)


# Tính Utility Score bằng trung bình của 4 tỷ lệ giữ lại
# Chỉ số này dùng để hỗ trợ so sánh các strategy trong đồ án
utility_score = (
    accuracy_retention
    + f1_retention
    + balanced_accuracy_retention
    + roc_auc_retention
) / 4


# Hiển thị Utility Score của strategy hiện tại
print(
    "Utility Score:",
    round(utility_score, 2),
    "%"
)

Utility Score: 97.13 %


In [40]:
# Gắn Utility Score vừa tính vào bảng tổng hợp strategy hiện tại
current_strategy_summary["Utility_Score_%"] = round(
    utility_score,
    2
)


# Hiển thị lại đầy đủ kết quả Privacy + Utility của strategy hiện tại
print(
    current_strategy_summary.to_string(
        index=False
    )
)

                       Strategy  k_min  l_min  Suppression_%  Accuracy_Original_%  Accuracy_Anonymized_%  Accuracy_Retention_%  F1_Retention_%  Balanced_Accuracy_Retention_%  ROC_AUC_Retention_%  Utility_Score_%
Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01                82.16                  81.31                 98.97           94.47                          97.09                97.98            97.13


In [41]:
# Tạo bảng lưu các strategy ứng viên
# Strategy hiện tại được đăng ký làm ứng viên đầu tiên
strategy_candidates = current_strategy_summary.copy()


# Kiểm tra strategy có đạt yêu cầu Privacy k=5 và l=2 hay không
strategy_candidates["Privacy_Valid"] = (
    (strategy_candidates["k_min"] >= 5)
    & (strategy_candidates["l_min"] >= 2)
)


# Hiển thị strategy ứng viên đầu tiên
print(
    strategy_candidates.to_string(
        index=False
    )
)

                       Strategy  k_min  l_min  Suppression_%  Accuracy_Original_%  Accuracy_Anonymized_%  Accuracy_Retention_%  F1_Retention_%  Balanced_Accuracy_Retention_%  ROC_AUC_Retention_%  Utility_Score_%  Privacy_Valid
Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01                82.16                  81.31                 98.97           94.47                          97.09                97.98            97.13           True


In [42]:
# Tính tỷ lệ Accuracy còn giữ lại trên tập Validation
validation_accuracy_retention = (
    accuracy_anonymized
    / accuracy_original
    * 100
)


# Tính tỷ lệ F1-score còn giữ lại trên tập Validation
validation_f1_retention = (
    f1_anonymized
    / f1_original
    * 100
)


# Tính tỷ lệ Balanced Accuracy còn giữ lại trên tập Validation
validation_balanced_accuracy_retention = (
    balanced_accuracy_anonymized
    / balanced_accuracy_original
    * 100
)


# Tính tỷ lệ ROC-AUC còn giữ lại trên tập Validation
validation_roc_auc_retention = (
    roc_auc_anonymized
    / roc_auc_original
    * 100
)


# Tính Utility Score dùng riêng cho việc lựa chọn strategy
# Không sử dụng kết quả Test trong quá trình xếp hạng strategy
validation_utility_score = (
    validation_accuracy_retention
    + validation_f1_retention
    + validation_balanced_accuracy_retention
    + validation_roc_auc_retention
) / 4


# Hiển thị Utility Score trên Validation
print(
    "Validation Utility Score:",
    round(validation_utility_score, 2),
    "%"
)

Validation Utility Score: 97.41 %


In [43]:
# Gắn Utility Score trên Validation vào strategy ứng viên hiện tại
# Đây mới là Utility Score được phép dùng để xếp hạng và chọn strategy
strategy_candidates["Validation_Utility_Score_%"] = round(
    validation_utility_score,
    2
)


# Đổi tên Utility_Score_% hiện tại để ghi rõ đây là kết quả trên tập Test
# Test chỉ dùng để báo cáo cuối, không dùng để lựa chọn strategy
strategy_candidates = strategy_candidates.rename(
    columns={
        "Utility_Score_%": "Test_Utility_Score_%"
    }
)


# Hiển thị lại bảng ứng viên số 1
print(
    strategy_candidates.to_string(
        index=False
    )
)

                       Strategy  k_min  l_min  Suppression_%  Accuracy_Original_%  Accuracy_Anonymized_%  Accuracy_Retention_%  F1_Retention_%  Balanced_Accuracy_Retention_%  ROC_AUC_Retention_%  Test_Utility_Score_%  Privacy_Valid  Validation_Utility_Score_%
Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01                82.16                  81.31                 98.97           94.47                          97.09                97.98                 97.13           True                       97.41


In [44]:
# Tạo đường dẫn đến dữ liệu ẩn danh của Strategy 2
strategy2_data_path = Path(
    "../data/processed_da_xu_li/anonymized_strategy2_k5_l2.csv"
)


# Tạo đường dẫn đến file chứa chỉ số gốc của 28.335 bản ghi Strategy 2
# File chỉ số này giúp tìm đúng các dòng tương ứng trong dataset gốc
strategy2_indices_path = Path(
    "../data/processed_da_xu_li/retained_row_indices_strategy2_k5_l2.csv"
)


# Đọc dữ liệu ẩn danh của Strategy 2 vào RAM
data_anonymized_strategy2 = pd.read_csv(
    strategy2_data_path,
    encoding="utf-8-sig"
)


# Đọc danh sách chỉ số các bản ghi được giữ lại của Strategy 2
strategy2_indices_df = pd.read_csv(
    strategy2_indices_path,
    encoding="utf-8-sig"
)


# Kiểm tra kích thước dữ liệu ẩn danh Strategy 2
print(
    "Kích thước dữ liệu ẩn danh Strategy 2:",
    data_anonymized_strategy2.shape
)


# Kiểm tra số lượng chỉ số gốc đã đọc
print(
    "Số chỉ số Strategy 2:",
    len(strategy2_indices_df)
)

Kích thước dữ liệu ẩn danh Strategy 2: (28335, 8)
Số chỉ số Strategy 2: 28335


In [45]:
# Lấy cột chứa chỉ số gốc của 28.335 bản ghi còn lại trong Strategy 2
# astype(int) đảm bảo các chỉ số được đưa về kiểu số nguyên
strategy2_indices = (
    strategy2_indices_df["original_row_index"]
    .astype(int)
)


# Dựa vào các chỉ số vừa đọc để lấy đúng 28.335 dòng tương ứng
# từ dataset gốc ban đầu
data_original_strategy2_matched = (
    data_original
    .loc[strategy2_indices]
    .copy()
)


# Đặt lại index từ 0, 1, 2, ...
# Giúp dữ liệu gốc và dữ liệu ẩn danh có cùng cách đánh số dòng
data_original_strategy2_matched = (
    data_original_strategy2_matched
    .reset_index(drop=True)
)


# Đặt lại index cho dữ liệu ẩn danh Strategy 2
# Không thay đổi nội dung dữ liệu, chỉ đánh số dòng lại từ đầu
data_anonymized_strategy2 = (
    data_anonymized_strategy2
    .reset_index(drop=True)
)


# Kiểm tra kích thước dữ liệu gốc đã lấy đúng theo Strategy 2
print(
    "Dữ liệu gốc Strategy 2 đã khớp:",
    data_original_strategy2_matched.shape
)


# Kiểm tra kích thước dữ liệu ẩn danh Strategy 2
print(
    "Dữ liệu ẩn danh Strategy 2:",
    data_anonymized_strategy2.shape
)

Dữ liệu gốc Strategy 2 đã khớp: (28335, 15)
Dữ liệu ẩn danh Strategy 2: (28335, 8)


In [46]:
# Chuẩn hóa cột thu_nhap của dữ liệu gốc Strategy 2
# strip() giúp loại bỏ khoảng trắng thừa nếu có
target_original_strategy2 = (
    data_original_strategy2_matched["thu_nhap"]
    .astype(str)
    .str.strip()
)


# Chuẩn hóa cột thu_nhap của dữ liệu ẩn danh Strategy 2
# Làm giống dữ liệu gốc để việc so sánh được chính xác
target_anonymized_strategy2 = (
    data_anonymized_strategy2["thu_nhap"]
    .astype(str)
    .str.strip()
)


# So sánh thu_nhap của từng dòng giữa hai dataset
# True nghĩa là dòng đó khớp, False nghĩa là không khớp
target_match_strategy2 = (
    target_original_strategy2
    == target_anonymized_strategy2
)


# Đếm số dòng có thu_nhap khớp nhau
so_dong_khop_strategy2 = int(
    target_match_strategy2.sum()
)


# Đếm số dòng có thu_nhap không khớp nhau
so_dong_khong_khop_strategy2 = int(
    (~target_match_strategy2).sum()
)


# Hiển thị kết quả kiểm tra
print(
    "Số dòng Strategy 2 có thu_nhap khớp:",
    so_dong_khop_strategy2
)

print(
    "Số dòng Strategy 2 có thu_nhap không khớp:",
    so_dong_khong_khop_strategy2
)

print(
    "Tỷ lệ khớp:",
    round(
        target_match_strategy2.mean() * 100,
        2
    ),
    "%"
)

Số dòng Strategy 2 có thu_nhap khớp: 28335
Số dòng Strategy 2 có thu_nhap không khớp: 0
Tỷ lệ khớp: 100.0 %


In [47]:
# Khai báo 7 thuộc tính gốc tương ứng với các QI của Strategy 2
# Dữ liệu gốc vẫn giữ thông tin chi tiết trước khi generalization
original_feature_columns_strategy2 = [
    "tuoi",
    "loai_cong_viec",
    "so_nam_hoc",
    "tinh_trang_hon_nhan",
    "nghe_nghiep",
    "gio_lam_moi_tuan",
    "quoc_gia"
]


# Khai báo 7 thuộc tính đã được ẩn danh của Strategy 2
# Strategy 2 dùng tuoi_nhom_high thay vì tuoi_nhom_3 của Strategy 1
anonymized_feature_columns_strategy2 = [
    "tuoi_nhom_high",
    "loai_cong_viec_nhom",
    "hoc_van_nhom_3",
    "hon_nhan_nhom_2",
    "nghe_nghiep_nhom_4",
    "gio_lam_nhom_3",
    "quoc_gia_nhom_2"
]


# Tạo tập đặc trưng X từ dữ liệu gốc tương ứng với Strategy 2
# Chỉ lấy đúng 7 loại thông tin dùng để so sánh với dữ liệu ẩn danh
X_original_strategy2 = (
    data_original_strategy2_matched[
        original_feature_columns_strategy2
    ]
    .copy()
)


# Tạo tập đặc trưng X từ dữ liệu ẩn danh Strategy 2
# Đây là phiên bản đã generalization của 7 thuộc tính tương ứng
X_anonymized_strategy2 = (
    data_anonymized_strategy2[
        anonymized_feature_columns_strategy2
    ]
    .copy()
)


# Lấy thu_nhap làm nhãn cần dự đoán
# strip() loại bỏ khoảng trắng thừa để nhãn được thống nhất
y_strategy2 = (
    data_original_strategy2_matched["thu_nhap"]
    .astype(str)
    .str.strip()
)


# Kiểm tra kích thước hai tập đặc trưng và nhãn
print(
    "X_original_strategy2:",
    X_original_strategy2.shape
)

print(
    "X_anonymized_strategy2:",
    X_anonymized_strategy2.shape
)

print(
    "y_strategy2:",
    y_strategy2.shape
)

X_original_strategy2: (28335, 7)
X_anonymized_strategy2: (28335, 7)
y_strategy2: (28335,)


In [48]:
# Import hàm dùng để chia dữ liệu
# Nếu đã import ở cell trước thì import lại cũng không sao
from sklearn.model_selection import train_test_split


# Lấy toàn bộ vị trí của 28.335 bản ghi Strategy 2
# Ta chia theo chỉ số để dữ liệu gốc và dữ liệu ẩn danh dùng đúng cùng các dòng
all_indices_strategy2 = X_original_strategy2.index


# Chia lần 1:
# 60% bản ghi dùng làm Train
# 40% còn lại tạm thời dành cho Validation và Test
train_indices_strategy2, temp_indices_strategy2 = train_test_split(
    all_indices_strategy2,
    test_size=0.40,
    random_state=42,
    stratify=y_strategy2
)


# Chia 40% còn lại thành hai phần bằng nhau
# Kết quả cuối cùng: 20% Validation và 20% Test
val_indices_strategy2, test_indices_strategy2 = train_test_split(
    temp_indices_strategy2,
    test_size=0.50,
    random_state=42,
    stratify=y_strategy2.loc[temp_indices_strategy2]
)


# Kiểm tra số lượng bản ghi trong tập Train
print(
    "Số bản ghi Train Strategy 2:",
    len(train_indices_strategy2)
)


# Kiểm tra số lượng bản ghi trong tập Validation
print(
    "Số bản ghi Validation Strategy 2:",
    len(val_indices_strategy2)
)


# Kiểm tra số lượng bản ghi trong tập Test
print(
    "Số bản ghi Test Strategy 2:",
    len(test_indices_strategy2)
)


# Kiểm tra tổng số bản ghi sau khi chia
# Tổng phải bằng đúng 28.335
print(
    "Tổng:",
    len(train_indices_strategy2)
    + len(val_indices_strategy2)
    + len(test_indices_strategy2)
)

Số bản ghi Train Strategy 2: 17001
Số bản ghi Validation Strategy 2: 5667
Số bản ghi Test Strategy 2: 5667
Tổng: 28335


Strategy 2 = phương án ẩn danh thứ hai, sử dụng một cấu hình generalization và suppression cụ thể để làm dữ liệu đạt k-anonymity và l-diversity.

In [49]:
#Tạo tập dữ liệu cho STRATEGY 2

# Lấy các dòng Train từ dữ liệu gốc
# Dùng đúng train_indices_strategy2 đã chia ở bước trước
X_original_strategy2_train = (
    X_original_strategy2
    .loc[train_indices_strategy2]
    .copy()
)


# Lấy các dòng Validation từ dữ liệu gốc
X_original_strategy2_val = (
    X_original_strategy2
    .loc[val_indices_strategy2]
    .copy()
)


# Lấy các dòng Test từ dữ liệu gốc
X_original_strategy2_test = (
    X_original_strategy2
    .loc[test_indices_strategy2]
    .copy()
)


# Lấy các dòng Train từ dữ liệu ẩn danh Strategy 2
# Dùng cùng chỉ số với dữ liệu gốc để hai bên khớp đúng từng bản ghi
X_anonymized_strategy2_train = (
    X_anonymized_strategy2
    .loc[train_indices_strategy2]
    .copy()
)


# Lấy các dòng Validation từ dữ liệu ẩn danh Strategy 2
X_anonymized_strategy2_val = (
    X_anonymized_strategy2
    .loc[val_indices_strategy2]
    .copy()
)


# Lấy các dòng Test từ dữ liệu ẩn danh Strategy 2
X_anonymized_strategy2_test = (
    X_anonymized_strategy2
    .loc[test_indices_strategy2]
    .copy()
)


# Tạo nhãn thu_nhap cho tập Train
y_strategy2_train = (
    y_strategy2
    .loc[train_indices_strategy2]
    .copy()
)


# Tạo nhãn thu_nhap cho tập Validation
y_strategy2_val = (
    y_strategy2
    .loc[val_indices_strategy2]
    .copy()
)


# Tạo nhãn thu_nhap cho tập Test
y_strategy2_test = (
    y_strategy2
    .loc[test_indices_strategy2]
    .copy()
)


# Kiểm tra kích thước tập Train của hai phía
print("TRAIN:")
print(
    "X_original_strategy2_train:",
    X_original_strategy2_train.shape
)
print(
    "X_anonymized_strategy2_train:",
    X_anonymized_strategy2_train.shape
)
print(
    "y_strategy2_train:",
    y_strategy2_train.shape
)


# Kiểm tra kích thước tập Validation
print("\nVALIDATION:")
print(
    "X_original_strategy2_val:",
    X_original_strategy2_val.shape
)
print(
    "X_anonymized_strategy2_val:",
    X_anonymized_strategy2_val.shape
)
print(
    "y_strategy2_val:",
    y_strategy2_val.shape
)


# Kiểm tra kích thước tập Test
print("\nTEST:")
print(
    "X_original_strategy2_test:",
    X_original_strategy2_test.shape
)
print(
    "X_anonymized_strategy2_test:",
    X_anonymized_strategy2_test.shape
)
print(
    "y_strategy2_test:",
    y_strategy2_test.shape
)

TRAIN:
X_original_strategy2_train: (17001, 7)
X_anonymized_strategy2_train: (17001, 7)
y_strategy2_train: (17001,)

VALIDATION:
X_original_strategy2_val: (5667, 7)
X_anonymized_strategy2_val: (5667, 7)
y_strategy2_val: (5667,)

TEST:
X_original_strategy2_test: (5667, 7)
X_anonymized_strategy2_test: (5667, 7)
y_strategy2_test: (5667,)


In [50]:
# Import numpy để dùng np.nan biểu diễn dữ liệu thiếu
# np.nan tương thích tốt với SimpleImputer của scikit-learn
import numpy as np


# Thay dấu ? bằng np.nan trong tập Train của dữ liệu gốc Strategy 2
# Chưa điền giá trị thiếu ở đây, chỉ chuẩn hóa cách biểu diễn missing value
X_original_strategy2_train = (
    X_original_strategy2_train
    .replace("?", np.nan)
)


# Thay dấu ? bằng np.nan trong tập Validation của dữ liệu gốc Strategy 2
# Không học thông tin từ Validation nên không gây data leakage
X_original_strategy2_val = (
    X_original_strategy2_val
    .replace("?", np.nan)
)


# Thay dấu ? bằng np.nan trong tập Test của dữ liệu gốc Strategy 2
# Test cũng chỉ được chuẩn hóa ký hiệu dữ liệu thiếu, chưa dùng để học
X_original_strategy2_test = (
    X_original_strategy2_test
    .replace("?", np.nan)
)


# Kiểm tra số lượng giá trị thiếu trong từng cột của tập Train
# Mục đích là xác nhận dấu ? đã được chuyển thành missing value thành công
print(
    X_original_strategy2_train
    .isna()
    .sum()
)

tuoi                     0
loai_cong_viec         542
so_nam_hoc               0
tinh_trang_hon_nhan      0
nghe_nghiep            542
gio_lam_moi_tuan         0
quoc_gia               231
dtype: int64


In [51]:
# Import clone để tạo bản sao sạch của pipeline đã có
# Bản sao này chưa học dữ liệu nào của Strategy 1
from sklearn.base import clone

# Import ColumnTransformer để xử lý các nhóm cột khác nhau
from sklearn.compose import ColumnTransformer


# Tạo bộ tiền xử lý riêng cho dữ liệu gốc của Strategy 2
# Dữ liệu gốc vẫn có cùng 3 cột số và 4 cột phân loại như Strategy 1
strategy2_original_preprocessor = clone(
    original_preprocessor
)


# Khai báo 7 cột phân loại của dữ liệu ẩn danh Strategy 2
# Điểm khác Strategy 1 là dùng tuoi_nhom_high thay cho tuoi_nhom_3
anonymized_categorical_columns_strategy2 = [
    "tuoi_nhom_high",
    "loai_cong_viec_nhom",
    "hoc_van_nhom_3",
    "hon_nhan_nhom_2",
    "nghe_nghiep_nhom_4",
    "gio_lam_nhom_3",
    "quoc_gia_nhom_2"
]


# Tạo bộ tiền xử lý riêng cho dữ liệu ẩn danh Strategy 2
# Dùng lại quy trình: điền missing + One-Hot Encoding
strategy2_anonymized_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",

            # clone() tạo bản sao sạch của pipeline categorical
            # để Strategy 2 tự học từ tập Train của chính nó
            clone(anonymized_categorical_pipeline),

            # Áp dụng pipeline cho 7 cột ẩn danh của Strategy 2
            anonymized_categorical_columns_strategy2
        )
    ]
)


# In hai bộ tiền xử lý để kiểm tra đã tạo thành công
print("PREPROCESSOR DỮ LIỆU GỐC - STRATEGY 2:")
print(strategy2_original_preprocessor)

print("\nPREPROCESSOR DỮ LIỆU ẨN DANH - STRATEGY 2:")
print(strategy2_anonymized_preprocessor)

PREPROCESSOR DỮ LIỆU GỐC - STRATEGY 2:
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tuoi', 'so_nam_hoc', 'gio_lam_moi_tuan']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['loai_cong_viec', 'tinh_trang_hon_nhan',
                                  'nghe_nghiep', 'quoc_gia'])])

PREPROCESSOR DỮ LIỆU ẨN DANH - STRATEGY 2:
ColumnTransformer(transformers=[('categorical',
                           

In [52]:
# TIỀN XỬ LÝ TRAIN / VALIDATION / TEST STRATEGY 2


# Cho preprocessor của dữ liệu gốc học trên tập Train Strategy 2
# fit_transform() vừa học quy tắc xử lý, vừa biến đổi dữ liệu Train
X_original_strategy2_train_processed = (
    strategy2_original_preprocessor
    .fit_transform(X_original_strategy2_train)
)


# Áp dụng đúng quy tắc đã học từ Train cho Validation
# Chỉ dùng transform() để tránh data leakage từ Validation
X_original_strategy2_val_processed = (
    strategy2_original_preprocessor
    .transform(X_original_strategy2_val)
)


# Áp dụng quy tắc đã học từ Train cho Test
# Test tuyệt đối không được dùng để học cách tiền xử lý
X_original_strategy2_test_processed = (
    strategy2_original_preprocessor
    .transform(X_original_strategy2_test)
)


# Cho preprocessor dữ liệu ẩn danh học trên Train Strategy 2
# Pipeline sẽ học các nhóm categorical chỉ từ tập Train
X_anonymized_strategy2_train_processed = (
    strategy2_anonymized_preprocessor
    .fit_transform(X_anonymized_strategy2_train)
)


# Áp dụng quy tắc đã học từ Train cho Validation của dữ liệu ẩn danh
X_anonymized_strategy2_val_processed = (
    strategy2_anonymized_preprocessor
    .transform(X_anonymized_strategy2_val)
)


# Áp dụng quy tắc đã học từ Train cho Test của dữ liệu ẩn danh
X_anonymized_strategy2_test_processed = (
    strategy2_anonymized_preprocessor
    .transform(X_anonymized_strategy2_test)
)


# Kiểm tra kích thước dữ liệu gốc sau tiền xử lý
# Train, Validation và Test phải có cùng số cột
print("DỮ LIỆU GỐC STRATEGY 2:")
print(
    "Train:",
    X_original_strategy2_train_processed.shape
)
print(
    "Validation:",
    X_original_strategy2_val_processed.shape
)
print(
    "Test:",
    X_original_strategy2_test_processed.shape
)


# Kiểm tra kích thước dữ liệu ẩn danh sau tiền xử lý
# Train, Validation và Test cũng phải có cùng số cột
print("\nDỮ LIỆU ẨN DANH STRATEGY 2:")
print(
    "Train:",
    X_anonymized_strategy2_train_processed.shape
)
print(
    "Validation:",
    X_anonymized_strategy2_val_processed.shape
)
print(
    "Test:",
    X_anonymized_strategy2_test_processed.shape
)

DỮ LIỆU GỐC STRATEGY 2:
Train: (17001, 71)
Validation: (5667, 71)
Test: (5667, 71)

DỮ LIỆU ẨN DANH STRATEGY 2:
Train: (17001, 22)
Validation: (5667, 22)
Test: (5667, 22)


Logistic Regression trên dữ liệu gốc của Strategy 2

In [53]:
# Import Logistic Regression để huấn luyện mô hình dự đoán thu_nhap
from sklearn.linear_model import LogisticRegression

# Import các metric để đánh giá mô hình trên tập Validation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)


# Tạo mô hình Logistic Regression cho dữ liệu gốc của Strategy 2
# Dùng cùng cấu hình với Strategy 1 để việc so sánh được công bằng
strategy2_original_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Huấn luyện mô hình bằng tập Train của dữ liệu gốc Strategy 2
strategy2_original_logistic_model.fit(
    X_original_strategy2_train_processed,
    y_strategy2_train
)


# Dự đoán nhãn thu_nhap trên tập Validation
# Validation dùng để đánh giá và so sánh strategy, chưa sử dụng Test
y_val_pred_strategy2_original = (
    strategy2_original_logistic_model.predict(
        X_original_strategy2_val_processed
    )
)


# Tính Accuracy: tỷ lệ dự đoán đúng trên toàn bộ Validation
accuracy_strategy2_original = accuracy_score(
    y_strategy2_val,
    y_val_pred_strategy2_original
)


# Tính Precision cho lớp >50K
# Cho biết khi mô hình dự đoán >50K thì có bao nhiêu dự đoán là đúng
precision_strategy2_original = precision_score(
    y_strategy2_val,
    y_val_pred_strategy2_original,
    pos_label=">50K"
)


# Tính Recall cho lớp >50K
# Cho biết trong số người thực sự >50K, mô hình tìm đúng được bao nhiêu
recall_strategy2_original = recall_score(
    y_strategy2_val,
    y_val_pred_strategy2_original,
    pos_label=">50K"
)


# Tính F1-score để cân bằng giữa Precision và Recall
f1_strategy2_original = f1_score(
    y_strategy2_val,
    y_val_pred_strategy2_original,
    pos_label=">50K"
)


# Tính Balanced Accuracy để đánh giá cân bằng giữa hai lớp thu nhập
balanced_accuracy_strategy2_original = balanced_accuracy_score(
    y_strategy2_val,
    y_val_pred_strategy2_original
)


# Tìm vị trí của lớp >50K trong danh sách lớp của mô hình
# Cần vị trí này để lấy đúng xác suất dự đoán cho lớp >50K
positive_class_index_strategy2_original = list(
    strategy2_original_logistic_model.classes_
).index(">50K")


# Lấy xác suất dự đoán thuộc lớp >50K trên tập Validation
y_val_prob_strategy2_original = (
    strategy2_original_logistic_model.predict_proba(
        X_original_strategy2_val_processed
    )[:, positive_class_index_strategy2_original]
)


# Tính ROC-AUC để đo khả năng phân biệt <=50K và >50K
roc_auc_strategy2_original = roc_auc_score(
    (y_strategy2_val == ">50K").astype(int),
    y_val_prob_strategy2_original
)


# Hiển thị 6 chỉ số Validation của dữ liệu gốc Strategy 2
print("LOGISTIC REGRESSION - STRATEGY 2 - DỮ LIỆU GỐC")
print(
    "Accuracy:",
    round(accuracy_strategy2_original, 4)
)
print(
    "Precision:",
    round(precision_strategy2_original, 4)
)
print(
    "Recall:",
    round(recall_strategy2_original, 4)
)
print(
    "F1-score:",
    round(f1_strategy2_original, 4)
)
print(
    "Balanced Accuracy:",
    round(balanced_accuracy_strategy2_original, 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_strategy2_original, 4)
)

LOGISTIC REGRESSION - STRATEGY 2 - DỮ LIỆU GỐC
Accuracy: 0.8115
Precision: 0.6848
Recall: 0.5604
F1-score: 0.6164
Balanced Accuracy: 0.7325
ROC-AUC: 0.8639


In [54]:
# LOGISTIC REGRESSION - DỮ LIỆU ẨN DANH STRATEGY 2


# Tạo một mô hình Logistic Regression mới cho dữ liệu ẩn danh Strategy 2
# Dùng cùng cấu hình với dữ liệu gốc để việc so sánh được công bằng
strategy2_anonymized_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Huấn luyện mô hình bằng tập Train của dữ liệu ẩn danh Strategy 2
strategy2_anonymized_logistic_model.fit(
    X_anonymized_strategy2_train_processed,
    y_strategy2_train
)


# Dự đoán thu_nhap trên tập Validation của Strategy 2
# Chưa sử dụng tập Test ở bước lựa chọn strategy
y_val_pred_strategy2_anonymized = (
    strategy2_anonymized_logistic_model.predict(
        X_anonymized_strategy2_val_processed
    )
)


# Tính Accuracy trên toàn bộ tập Validation
accuracy_strategy2_anonymized = accuracy_score(
    y_strategy2_val,
    y_val_pred_strategy2_anonymized
)


# Tính Precision cho lớp >50K
# Cho biết trong các trường hợp mô hình dự đoán >50K,
# có bao nhiêu trường hợp thực sự đúng
precision_strategy2_anonymized = precision_score(
    y_strategy2_val,
    y_val_pred_strategy2_anonymized,
    pos_label=">50K"
)


# Tính Recall cho lớp >50K
# Cho biết trong những người thực sự >50K,
# mô hình phát hiện đúng được bao nhiêu
recall_strategy2_anonymized = recall_score(
    y_strategy2_val,
    y_val_pred_strategy2_anonymized,
    pos_label=">50K"
)


# Tính F1-score để cân bằng giữa Precision và Recall
f1_strategy2_anonymized = f1_score(
    y_strategy2_val,
    y_val_pred_strategy2_anonymized,
    pos_label=">50K"
)


# Tính Balanced Accuracy để đánh giá cân bằng giữa hai lớp thu nhập
balanced_accuracy_strategy2_anonymized = balanced_accuracy_score(
    y_strategy2_val,
    y_val_pred_strategy2_anonymized
)


# Tìm vị trí của lớp >50K trong danh sách lớp của mô hình
# Cần vị trí này để lấy đúng xác suất dự đoán lớp >50K
positive_class_index_strategy2_anonymized = list(
    strategy2_anonymized_logistic_model.classes_
).index(">50K")


# Lấy xác suất dự đoán thuộc lớp >50K trên tập Validation
y_val_prob_strategy2_anonymized = (
    strategy2_anonymized_logistic_model.predict_proba(
        X_anonymized_strategy2_val_processed
    )[:, positive_class_index_strategy2_anonymized]
)


# Tính ROC-AUC để đo khả năng phân biệt hai lớp <=50K và >50K
roc_auc_strategy2_anonymized = roc_auc_score(
    (y_strategy2_val == ">50K").astype(int),
    y_val_prob_strategy2_anonymized
)


# Hiển thị 6 chỉ số Validation của dữ liệu ẩn danh Strategy 2
print("LOGISTIC REGRESSION - STRATEGY 2 - DỮ LIỆU ẨN DANH")
print(
    "Accuracy:",
    round(accuracy_strategy2_anonymized, 4)
)
print(
    "Precision:",
    round(precision_strategy2_anonymized, 4)
)
print(
    "Recall:",
    round(recall_strategy2_anonymized, 4)
)
print(
    "F1-score:",
    round(f1_strategy2_anonymized, 4)
)
print(
    "Balanced Accuracy:",
    round(balanced_accuracy_strategy2_anonymized, 4)
)
print(
    "ROC-AUC:",
    round(roc_auc_strategy2_anonymized, 4)
)

LOGISTIC REGRESSION - STRATEGY 2 - DỮ LIỆU ẨN DANH
Accuracy: 0.805
Precision: 0.6895
Recall: 0.5062
F1-score: 0.5838
Balanced Accuracy: 0.7109
ROC-AUC: 0.8541


In [55]:
# Tính tỷ lệ Accuracy còn giữ lại sau khi ẩn danh
# Lấy Accuracy của dữ liệu ẩn danh chia cho Accuracy của dữ liệu gốc
accuracy_retention_strategy2 = (
    accuracy_strategy2_anonymized
    / accuracy_strategy2_original
    * 100
)


# Tính tỷ lệ F1-score còn giữ lại sau khi ẩn danh
# F1-score giúp cân bằng giữa Precision và Recall
f1_retention_strategy2 = (
    f1_strategy2_anonymized
    / f1_strategy2_original
    * 100
)


# Tính tỷ lệ Balanced Accuracy còn giữ lại
# Chỉ số này hữu ích khi hai lớp thu nhập không hoàn toàn cân bằng
balanced_accuracy_retention_strategy2 = (
    balanced_accuracy_strategy2_anonymized
    / balanced_accuracy_strategy2_original
    * 100
)


# Tính tỷ lệ ROC-AUC còn giữ lại
# ROC-AUC cho biết khả năng phân biệt hai lớp thu nhập của mô hình
roc_auc_retention_strategy2 = (
    roc_auc_strategy2_anonymized
    / roc_auc_strategy2_original
    * 100
)


# Tính Validation Utility Score của Strategy 2
# Lấy trung bình 4 tỷ lệ giữ lại giống đúng cách đã dùng cho Strategy 1
# Đây là chỉ số do đồ án xây dựng để hỗ trợ so sánh các strategy
validation_utility_score_strategy2 = (
    accuracy_retention_strategy2
    + f1_retention_strategy2
    + balanced_accuracy_retention_strategy2
    + roc_auc_retention_strategy2
) / 4


# Hiển thị từng tỷ lệ giữ lại để dễ kiểm tra
print(
    "Accuracy Retention Strategy 2:",
    round(accuracy_retention_strategy2, 2),
    "%"
)

print(
    "F1 Retention Strategy 2:",
    round(f1_retention_strategy2, 2),
    "%"
)

print(
    "Balanced Accuracy Retention Strategy 2:",
    round(balanced_accuracy_retention_strategy2, 2),
    "%"
)

print(
    "ROC-AUC Retention Strategy 2:",
    round(roc_auc_retention_strategy2, 2),
    "%"
)


# Hiển thị Utility Score dùng để xếp hạng Strategy 2
print(
    "\nValidation Utility Score Strategy 2:",
    round(validation_utility_score_strategy2, 2),
    "%"
)

Accuracy Retention Strategy 2: 99.2 %
F1 Retention Strategy 2: 94.72 %
Balanced Accuracy Retention Strategy 2: 97.06 %
ROC-AUC Retention Strategy 2: 98.86 %

Validation Utility Score Strategy 2: 97.46 %


In [56]:
# SO SÁNH STRATEGY 1 VÀ STRATEGY 2 TRÊN VALIDATION


# Tạo bảng chỉ chứa các thông tin được phép dùng để lựa chọn strategy
# Không đưa kết quả Test vào vì Test chỉ dành cho đánh giá cuối cùng
strategy_selection_table = pd.DataFrame({
    # Tên của hai strategy đang được so sánh
    "Strategy": [
        "Country2_Hours3_Age3_Edu3_k5_l2",
        "Country2_Hours3_AgeHigh_Edu3_k5_l2"
    ],

    # Cả hai strategy đều đã được kiểm tra đạt k=5
    "k_min": [
        5,
        5
    ],

    # Cả hai strategy đều đã được kiểm tra đạt l=2
    "l_min": [
        2,
        2
    ],

    # Tỷ lệ suppression của từng strategy
    # Strategy có suppression thấp hơn sẽ giữ lại nhiều dữ liệu hơn
    "Suppression_%": [
        6.01,
        12.98
    ],

    # Utility Score trên Validation
    # Đây là chỉ số được dùng để hỗ trợ lựa chọn strategy
    "Validation_Utility_Score_%": [
        validation_utility_score,
        validation_utility_score_strategy2
    ]
})


# Kiểm tra strategy có đạt yêu cầu Privacy hay không
# True nghĩa là đồng thời đạt k>=5 và l>=2
strategy_selection_table["Privacy_Valid"] = (
    (strategy_selection_table["k_min"] >= 5)
    & (strategy_selection_table["l_min"] >= 2)
)


# Làm tròn các số để bảng dễ đọc
strategy_selection_table[
    [
        "Suppression_%",
        "Validation_Utility_Score_%"
    ]
] = (
    strategy_selection_table[
        [
            "Suppression_%",
            "Validation_Utility_Score_%"
        ]
    ]
    .round(2)
)


# Hiển thị bảng so sánh hai strategy
print(
    strategy_selection_table.to_string(
        index=False
    )
)

                          Strategy  k_min  l_min  Suppression_%  Validation_Utility_Score_%  Privacy_Valid
   Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01                       97.41           True
Country2_Hours3_AgeHigh_Edu3_k5_l2      5      2          12.98                       97.46           True


Strategy Country2_Hours3_Age3_Edu3_k5_l2 được lựa chọn vì đáp ứng yêu cầu k-anonymity và l-diversity, đồng thời duy trì utility cao và có tỷ lệ suppression thấp hơn, thể hiện sự cân bằng tốt hơn giữa Privacy và Utility.

In [57]:
# Tính tỷ lệ dữ liệu còn được giữ lại sau suppression
# Ví dụ suppression 6.01% nghĩa là còn giữ lại 93.99% dữ liệu
strategy_selection_table["Data_Retention_%"] = (
    100
    - strategy_selection_table["Suppression_%"]
)


# Đặt trọng số cho hai tiêu chí lựa chọn
# 50% ưu tiên giữ lại nhiều dữ liệu
# 50% ưu tiên duy trì hiệu năng Machine Learning
weight_data_retention = 0.5
weight_ml_utility = 0.5


# Tính điểm lựa chọn tổng hợp cho mỗi strategy
# Chỉ các strategy đã đạt k=5 và l=2 mới có ý nghĩa để đem xếp hạng
strategy_selection_table["Selection_Score"] = (
    strategy_selection_table["Data_Retention_%"]
    * weight_data_retention
    +
    strategy_selection_table["Validation_Utility_Score_%"]
    * weight_ml_utility
)


# Làm tròn kết quả để bảng dễ đọc
strategy_selection_table[
    [
        "Data_Retention_%",
        "Selection_Score"
    ]
] = (
    strategy_selection_table[
        [
            "Data_Retention_%",
            "Selection_Score"
        ]
    ]
    .round(2)
)


# Chỉ lấy các strategy đã đạt điều kiện Privacy
valid_strategies = (
    strategy_selection_table[
        strategy_selection_table["Privacy_Valid"]
    ]
    .copy()
)


# Xếp hạng strategy theo Selection Score từ cao xuống thấp
strategy_ranking = (
    valid_strategies
    .sort_values(
        by="Selection_Score",
        ascending=False
    )
    .reset_index(drop=True)
)


# Hiển thị bảng xếp hạng strategy
print(
    strategy_ranking.to_string(
        index=False
    )
)

                          Strategy  k_min  l_min  Suppression_%  Validation_Utility_Score_%  Privacy_Valid  Data_Retention_%  Selection_Score
   Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01                       97.41           True             93.99            95.70
Country2_Hours3_AgeHigh_Edu3_k5_l2      5      2          12.98                       97.46           True             87.02            92.24


In [58]:
# Lấy dòng đầu tiên của bảng xếp hạng
# Vì strategy_ranking đã được sắp xếp Selection Score từ cao xuống thấp
best_strategy = strategy_ranking.iloc[0]


# Lấy tên strategy tốt nhất
# Giá trị này sau này có thể dùng để hệ thống tự quyết định phương án ẩn danh
selected_strategy_name = best_strategy["Strategy"]


# Lấy Selection Score của strategy tốt nhất
selected_strategy_score = best_strategy["Selection_Score"]


# Lấy tỷ lệ suppression của strategy được chọn
selected_strategy_suppression = best_strategy["Suppression_%"]


# Lấy Utility Score trên Validation của strategy được chọn
selected_strategy_utility = best_strategy[
    "Validation_Utility_Score_%"
]


# Hiển thị kết quả lựa chọn tự động
print(
    "Strategy được hệ thống lựa chọn:",
    selected_strategy_name
)

print(
    "Selection Score:",
    selected_strategy_score
)

print(
    "Suppression:",
    selected_strategy_suppression,
    "%"
)

print(
    "Validation Utility Score:",
    selected_strategy_utility,
    "%"
)

Strategy được hệ thống lựa chọn: Country2_Hours3_Age3_Edu3_k5_l2
Selection Score: 95.7
Suppression: 6.01 %
Validation Utility Score: 97.41 %


In [59]:
# Tạo bảng chỉ chứa strategy được hệ thống lựa chọn
# Bảng này sẽ dùng làm kết quả cuối của bước Select Optimal Strategy
selected_strategy_summary = strategy_ranking.iloc[[0]].copy()


# Thêm thứ hạng để thể hiện đây là strategy đứng đầu
selected_strategy_summary["Rank"] = 1


# Sắp xếp lại các cột để bảng dễ đọc trong báo cáo
selected_strategy_summary = selected_strategy_summary[
    [
        "Rank",
        "Strategy",
        "k_min",
        "l_min",
        "Suppression_%",
        "Data_Retention_%",
        "Validation_Utility_Score_%",
        "Selection_Score",
        "Privacy_Valid"
    ]
]


# Hiển thị kết quả cuối của bước lựa chọn strategy
print(
    selected_strategy_summary.to_string(
        index=False
    )
)

 Rank                        Strategy  k_min  l_min  Suppression_%  Data_Retention_%  Validation_Utility_Score_%  Selection_Score  Privacy_Valid
    1 Country2_Hours3_Age3_Edu3_k5_l2      5      2           6.01             93.99                       97.41             95.7           True


In [60]:
# Import sys để Python quản lý danh sách các thư mục dùng để tìm module
import sys

# Import Path để tạo đường dẫn đến thư mục src
from pathlib import Path

# Import importlib để có thể nạp lại file utility_evaluator.py mới nhất
import importlib


# Tạo đường dẫn tuyệt đối đến thư mục src
# Notebook đang ở thư mục notebooks nên ../src nghĩa là đi ra 1 cấp rồi vào src
src_path = Path("../src").resolve()


# Kiểm tra xem thư mục src đã có trong sys.path hay chưa
if str(src_path) not in sys.path:

    # Nếu chưa có thì thêm thư mục src vào danh sách tìm kiếm module
    # Nhờ vậy notebook mới import được utility_evaluator.py
    sys.path.append(
        str(src_path)
    )


# Import module utility_evaluator từ thư mục src
import utility_evaluator


# Nạp lại phiên bản mới nhất của utility_evaluator.py
# Điều này hữu ích khi chúng ta vừa sửa hoặc thêm code trong file
importlib.reload(
    utility_evaluator
)


# Gọi hàm evaluate_classification_metrics() trong file src
# Dùng kết quả Logistic Regression trên dữ liệu ẩn danh Strategy 2
utility_test_from_src = (
    utility_evaluator.evaluate_classification_metrics(

        # Nhãn thật của tập Validation Strategy 2
        y_true=y_strategy2_val,

        # Nhãn mà Logistic Regression đã dự đoán
        y_pred=y_val_pred_strategy2_anonymized,

        # Xác suất dự đoán thuộc lớp >50K
        # Giá trị này được dùng để tính ROC-AUC
        y_prob=y_val_prob_strategy2_anonymized,

        # Xác định >50K là lớp dương mà đồ án đang quan tâm
        positive_label=">50K"
    )
)


# Hiển thị kết quả do hàm trong src tính ra
print(
    utility_test_from_src
)

{'accuracy': 0.8050114699135344, 'precision': 0.6895017793594306, 'recall': 0.5062050947093403, 'f1_score': 0.583804143126177, 'balanced_accuracy': 0.7109120251109564, 'roc_auc': 0.8540626219952068}


In [61]:
# Import importlib để nạp lại phiên bản mới nhất của utility_evaluator.py
# Vì chúng ta vừa thêm hàm calculate_utility_retention()
import importlib


# Import module utility_evaluator trong thư mục src
import utility_evaluator


# Nạp lại file utility_evaluator.py
# Giúp notebook nhận được hàm mới vừa thêm
importlib.reload(
    utility_evaluator
)


# BƯỚC 1: TÍNH METRIC CỦA DỮ LIỆU GỐC STRATEGY 2


# Gọi hàm trong src để tính 6 chỉ số ML của dữ liệu gốc Strategy 2
original_metrics_strategy2_from_src = (
    utility_evaluator.evaluate_classification_metrics(

        # Nhãn thật của tập Validation Strategy 2
        y_true=y_strategy2_val,

        # Kết quả Logistic Regression dự đoán trên dữ liệu gốc
        y_pred=y_val_pred_strategy2_original,

        # Xác suất dự đoán lớp >50K trên dữ liệu gốc
        y_prob=y_val_prob_strategy2_original,

        # Xác định >50K là lớp dương
        positive_label=">50K"
    )
)


# BƯỚC 2: TÍNH METRIC CỦA DỮ LIỆU ẨN DANH STRATEGY 2


# Gọi cùng một hàm để tính 6 chỉ số ML của dữ liệu ẩn danh Strategy 2
anonymized_metrics_strategy2_from_src = (
    utility_evaluator.evaluate_classification_metrics(

        # Vẫn sử dụng cùng nhãn thật
        y_true=y_strategy2_val,

        # Kết quả Logistic Regression dự đoán trên dữ liệu ẩn danh
        y_pred=y_val_pred_strategy2_anonymized,

        # Xác suất dự đoán lớp >50K trên dữ liệu ẩn danh
        y_prob=y_val_prob_strategy2_anonymized,

        # Vẫn xác định >50K là lớp dương
        positive_label=">50K"
    )
)


# BƯỚC 3: TÍNH UTILITY RETENTION


# Đưa metric của dữ liệu gốc và dữ liệu ẩn danh vào hàm mới
# Hàm sẽ tự tính các tỷ lệ giữ lại và Utility Score
utility_retention_strategy2_from_src = (
    utility_evaluator.calculate_utility_retention(

        # Kết quả ML của dữ liệu gốc
        original_metrics=original_metrics_strategy2_from_src,

        # Kết quả ML của dữ liệu ẩn danh
        anonymized_metrics=anonymized_metrics_strategy2_from_src
    )
)


# Hiển thị kết quả Utility do code trong src tự tính
print(
    utility_retention_strategy2_from_src
)

{'accuracy_retention_%': 99.2, 'f1_retention_%': 94.72, 'balanced_accuracy_retention_%': 97.06, 'roc_auc_retention_%': 98.86, 'utility_score_%': 97.46}


In [62]:
# Import importlib để có thể nạp lại phiên bản mới nhất của strategy_selector.py
import importlib

# Import module strategy_selector nằm trong thư mục src
# Trước đó notebook đã thêm thư mục src vào sys.path nên có thể import trực tiếp
import strategy_selector


# Nạp lại file strategy_selector.py
# Việc này giúp notebook nhận đúng code mới nhất vừa lưu
importlib.reload(
    strategy_selector
)


# Gọi hàm calculate_selection_score() bằng kết quả thật của Strategy 1
# Strategy 1 có suppression = 6.01%
# Validation Utility Score = 97.41%
# Và đã đạt cả k=5, l=2 nên privacy_valid = True
selection_test_strategy1 = (
    strategy_selector.calculate_selection_score(

        # Tỷ lệ bản ghi bị loại của Strategy 1
        suppression_percent=6.01,

        # Utility Score trên Validation của Strategy 1
        utility_score_percent=97.41,

        # True vì Strategy 1 đã đạt k=5 và l=2
        privacy_valid=True,

        # Cho Data Retention trọng số 50%
        weight_data_retention=0.5,

        # Cho ML Utility trọng số 50%
        weight_utility=0.5
    )
)


# Hiển thị kết quả do hàm trong src tự tính
print(
    selection_test_strategy1
)

{'data_retention_%': 93.99, 'utility_score_%': 97.41, 'selection_score': 95.7, 'eligible': True}


In [63]:
# Import importlib để có thể nạp lại phiên bản mới nhất của strategy_selector.py
import importlib


# Import module strategy_selector từ thư mục src
import strategy_selector


# Nạp lại file strategy_selector.py
# Giúp notebook sử dụng đúng code mới nhất vừa thêm hàm rank_strategies()
importlib.reload(
    strategy_selector
)


# Tạo danh sách chứa 2 strategy thật mà đồ án đã đánh giá
strategies_test = [

    # Strategy 1:
    # Đã đạt k=5, l=2
    # Suppression = 6.01%
    # Validation Utility Score = 97.41%
    {
        "strategy": "Country2_Hours3_Age3_Edu3_k5_l2",
        "suppression_percent": 6.01,
        "utility_score_percent": 97.41,
        "privacy_valid": True
    },

    # Strategy 2:
    # Đã đạt k=5, l=2
    # Suppression = 12.98%
    # Validation Utility Score = 97.46%
    {
        "strategy": "Country2_Hours3_AgeHigh_Edu3_k5_l2",
        "suppression_percent": 12.98,
        "utility_score_percent": 97.46,
        "privacy_valid": True
    }
]


# Gọi hàm rank_strategies() trong file src
# Hàm sẽ tự:
# - tính Data Retention
# - tính Selection Score
# - sắp xếp từ điểm cao xuống thấp
# - gắn thứ hạng Rank 1, Rank 2
ranking_test_from_src = strategy_selector.rank_strategies(
    strategies=strategies_test,

    # Cho Data Retention trọng số 50%
    weight_data_retention=0.5,

    # Cho ML Utility trọng số 50%
    weight_utility=0.5
)


# Hiển thị từng strategy theo thứ hạng
for result in ranking_test_from_src:

    # In một dòng ngăn cách để kết quả dễ đọc
    print("-----------------------------------")

    # Hiển thị thứ hạng của strategy
    print(
        "Rank:",
        result["rank"]
    )

    # Hiển thị tên strategy
    print(
        "Strategy:",
        result["strategy"]
    )

    # Hiển thị tỷ lệ dữ liệu còn giữ lại
    print(
        "Data Retention:",
        result["data_retention_%"],
        "%"
    )

    # Hiển thị Utility Score trên Validation
    print(
        "Utility Score:",
        result["utility_score_%"],
        "%"
    )

    # Hiển thị Selection Score cuối cùng
    print(
        "Selection Score:",
        result["selection_score"]
    )

-----------------------------------
Rank: 1
Strategy: Country2_Hours3_Age3_Edu3_k5_l2
Data Retention: 93.99 %
Utility Score: 97.41 %
Selection Score: 95.7
-----------------------------------
Rank: 2
Strategy: Country2_Hours3_AgeHigh_Edu3_k5_l2
Data Retention: 87.02 %
Utility Score: 97.46 %
Selection Score: 92.24


In [64]:
# Import importlib để nạp lại phiên bản mới nhất của strategy_selector.py
import importlib


# Import module strategy_selector nằm trong thư mục src
import strategy_selector


# Nạp lại module để notebook nhận hàm select_best_strategy() vừa thêm
importlib.reload(
    strategy_selector
)


# Gọi hàm select_best_strategy() với 2 strategy thật đã tạo trước đó
# strategies_test đang chứa Strategy 1 và Strategy 2
best_strategy_from_src = (
    strategy_selector.select_best_strategy(

        # Danh sách 2 strategy cần hệ thống lựa chọn
        strategies=strategies_test,

        # Data Retention chiếm 50% Selection Score
        weight_data_retention=0.5,

        # ML Utility chiếm 50% Selection Score
        weight_utility=0.5
    )
)


# Hiển thị tên strategy mà hệ thống tự chọn
print(
    "Strategy được chọn:",
    best_strategy_from_src["strategy"]
)


# Hiển thị thứ hạng của strategy được chọn
print(
    "Rank:",
    best_strategy_from_src["rank"]
)


# Hiển thị tỷ lệ dữ liệu còn được giữ lại
print(
    "Data Retention:",
    best_strategy_from_src["data_retention_%"],
    "%"
)


# Hiển thị Utility Score trên Validation
print(
    "Utility Score:",
    best_strategy_from_src["utility_score_%"],
    "%"
)


# Hiển thị Selection Score dùng để lựa chọn
print(
    "Selection Score:",
    best_strategy_from_src["selection_score"]
)

Strategy được chọn: Country2_Hours3_Age3_Edu3_k5_l2
Rank: 1
Data Retention: 93.99 %
Utility Score: 97.41 %
Selection Score: 95.7


Hệ thống đã lựa chọn strategy Country2_Hours3_Age3_Edu3_k5_l2 ở vị trí Rank 1. Strategy này đạt yêu cầu Privacy k=5, l=2, giữ lại 93.99% dữ liệu và duy trì 97.41% Utility cho Machine Learning. Điểm lựa chọn tổng hợp đạt 95.7, cho thấy đây là phương án cân bằng tốt nhất giữa bảo mật và khả năng sử dụng dữ liệu trong các ứng viên hiện tại.